# Neural Scaling Laws — Kaggle Experiments (A–E)

Reproducible experiments accompanying the paper:  
**"Neural Scaling Laws: A Survey, Meta-Analysis, Unified Multi-Axis Framework, and Open Problems"**

### Overview
| Experiment | Script section | What it tests | Est. runtime |
|---|---|---|---|
| A | Exp A — Multi-model | P1: T* increases with difficulty | 7–8 h |
| B | Exp B — P6 scaling | P6: T* decreases with model size | 6–7 h |
| C | Exp C — Hurst exponent | P5: V(T) ∝ T^{2H}, H≈0.5 | 4–5 h |
| D | Exp D — Precision sweep | P3: T* monotone in precision | 5–6 h |
| E | Exp E — Prospective | Out-of-sample prediction test | 3–4 h |
| Merge | Final merge | Summary figures + tables | ~5 min |

**Hardware required:** Kaggle T4×2 (30 GB VRAM combined)  
**Run cells in order** — each experiment saves checkpoints so you can resume after a crash.  
All outputs go to `/kaggle/working/results/expA/` … `/kaggle/working/results/merged/`


## 0 · Setup — install packages and configure output paths

In [ ]:
import subprocess, sys

# Install required packages
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'accelerate', 'bitsandbytes', 'scipy', 'tqdm'])

import os, json, math, time, warnings, gc, re
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

warnings.filterwarnings('ignore')

# Output directories — one per experiment
DIRS = {
    'expA': Path('/kaggle/working/results/expA'),
    'expB': Path('/kaggle/working/results/expB'),
    'expC': Path('/kaggle/working/results/expC'),
    'expD': Path('/kaggle/working/results/expD'),
    'expE': Path('/kaggle/working/results/expE'),
    'merged': Path('/kaggle/working/results/merged'),
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Directories ready:")
for k, v in DIRS.items():
    print(f"  {k}: {v}")


## 1 · Shared helpers — g-model, fitting, bootstrap CI

In [ ]:
from scipy.optimize import curve_fit
from scipy.special import erfc

def g_model(V, delta, sigma_r2):
    """Expected reward under Gaussian competition: g(V) = φ(Δ/√(σ_r²+V))."""
    denom = np.sqrt(sigma_r2 + np.array(V, dtype=float))
    return 0.5 * erfc(-np.array(delta, dtype=float) / (np.sqrt(2) * denom))

def accuracy_curve(T, delta, sigma_r2, k):
    """Accuracy(T) = 100 * g(k*T; Δ, σ_r²)."""
    return 100.0 * g_model(k * np.array(T, dtype=float), delta, sigma_r2)

def fit_g_model(T_vals, acc_vals, n_starts=20):
    """Fit g-model to (T, accuracy%) data. Returns (params, T_star, r2) or None."""
    T = np.array(T_vals, dtype=float)
    A = np.array(acc_vals, dtype=float)
    best = None
    for _ in range(n_starts):
        p0 = [np.random.uniform(0.5, 3), np.random.uniform(0.1, 2),
              np.random.uniform(1e-5, 1e-3)]
        try:
            popt, _ = curve_fit(accuracy_curve, T, A, p0=p0,
                                bounds=([0,1e-9,1e-9],[10,10,0.1]),
                                maxfev=5000)
            res = A - accuracy_curve(T, *popt)
            ss_res, ss_tot = np.sum(res**2), np.sum((A - A.mean())**2)
            r2 = 1 - ss_res/ss_tot if ss_tot > 0 else 0
            score = -ss_res
            if best is None or score > best[0]:
                best = (score, popt, r2)
        except Exception:
            pass
    if best is None:
        return None
    _, popt, r2 = best
    delta, sigma_r2, k = popt
    T_star = (delta**2 - sigma_r2) / k if delta**2 > sigma_r2 else None
    return {'delta': float(delta), 'sigma_r2': float(sigma_r2), 'k': float(k),
            'T_star': float(T_star) if T_star else None, 'r2': float(r2)}

def bootstrap_tstar(T_vals, acc_vals, B=600, **kwargs):
    """Bootstrap 95% CI on T_star. Returns (lower, upper) or (None, None)."""
    stars = []
    for _ in range(B):
        idx = np.random.randint(0, len(T_vals), len(T_vals))
        Ts, As = np.array(T_vals)[idx], np.array(acc_vals)[idx]
        r = fit_g_model(Ts.tolist(), As.tolist())
        if r and r['T_star']:
            stars.append(r['T_star'])
    if len(stars) < 10:
        return None, None
    return float(np.percentile(stars, 2.5)), float(np.percentile(stars, 97.5))

print("Helpers loaded.")


---
## Experiment A — Multi-Model T* Survey

**Prediction P1 tested:** T*(difficulty=d) is monotone-increasing in d.  
Models: Phi-3.5-mini-instruct (3.8B), Qwen2.5-3B-Instruct (3.0B), Gemma-2-2b-it (2.0B)  
Checkpoints saved to `/kaggle/working/results/expA/checkpoint.json` — safe to restart.


In [ ]:
import os, sys, json, time, math, warnings, gc, re
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")
START_TIME = time.time()

# ── 0. INSTALL ────────────────────────────────────────────────────────────────
import subprocess
for pkg in ["torch", "transformers", "accelerate", "bitsandbytes",
            "scipy", "scikit-learn", "sympy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                   capture_output=True)

import numpy as np
import scipy.stats as stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import torch
import sympy as sp
from sympy.parsing.sympy_parser import (
    parse_expr, standard_transformations, implicit_multiplication_application,
)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

torch.manual_seed(42); np.random.seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Setup] device={DEVICE}  GPUs={torch.cuda.device_count()}")
if DEVICE == "cuda":
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB")

OUT = Path("/kaggle/working/results_part3")
OUT.mkdir(parents=True, exist_ok=True)

# ── 1. CONSTANTS ──────────────────────────────────────────────────────────────

BUDGETS   = [150, 300, 600, 1200, 2400, 4800]
N_SAMPLES = 2       # independent completions per (problem, budget)
N_PROBS   = 8       # problems per level

SESSION_LIMIT_MIN = 510   # warn / soft-stop at 8.5 h

MODELS = {
    "phi35":  "microsoft/Phi-3.5-mini-instruct",
    "qwen3b": "Qwen/Qwen2.5-3B-Instruct",
    "gemma2": "google/gemma-2-2b-it",
}

MODEL_BITS = {"phi35": 8, "qwen3b": 8, "gemma2": 8}

# Which models support a "system" turn in their chat template
MODEL_HAS_SYSTEM = {"phi35": True, "qwen3b": True, "gemma2": False}

SYS_PROMPT = (
    "You are a precise math assistant. "
    "Think step by step. "
    "Write your final numerical answer after the word ANSWER: on its own line."
)

MATH_PROBLEMS = {
    "Level_1": [
        ("Compute $2^3 + 3^2$.", "17"),
        ("What is $15\\% $ of $200$?", "30"),
        ("If $x + 5 = 12$, find $x$.", "7"),
        ("What is $\\sqrt{49}$?", "7"),
        ("Calculate $3!$.", "6"),
        ("What is $4^2 - 3^2$?", "7"),
        ("Find $x$: $2x = 18$.", "9"),
        ("Compute $7 \\times 8$.", "56"),
        ("What is $2^5$?", "32"),
        ("Find the mean of $4, 6, 8, 10$.", "7"),
        ("What is $(-3)^2$?", "9"),
        ("If $y - 4 = 10$, find $y$.", "14"),
        ("Simplify $12/18$.", "2/3"),
        ("Perimeter of a square with side 5.", "20"),
        ("What is $10\\%$ of $350$?", "35"),
    ],
    "Level_3": [
        ("Solve $x^2 - 5x + 6 = 0$.", "2 or 3"),
        ("A circle has radius 7. Area in terms of pi.", "49*pi"),
        ("If $f(x)=2x^2-3x+1$, find $f(3)$.", "10"),
        ("Solve: $\\log_2 8 = x$.", "3"),
        ("Distance between $(0,0)$ and $(3,4)$.", "5"),
        ("What is $C(6,2)$?", "15"),
        ("Find the 10th term of $3,7,11,\\dots$", "39"),
        ("Solve $|x-3|=5$.", "8 or -2"),
        ("Find the vertex of $y=x^2-4x+3$.", "2"),
        ("Arrangements: 4 books from 6?", "360"),
        ("Solve $2^x=32$.", "5"),
        ("Find $\\gcd(48,18)$.", "6"),
        ("P(A)=0.3, P(B)=0.5, independent. Find P(A and B).", "0.15"),
        ("Derivative of $x^3$.", "3*x**2"),
        ("Sum of first 10 natural numbers.", "55"),
    ],
    "Level_5": [
        ("Compute sum_{k=1}^{inf} 1/(k(k+1)).", "1"),
        ("Evaluate integral_0^pi sin^2(x) dx.", "pi/2"),
        ("Limit as x->0 of sin(3x)/x.", "3"),
        ("Eigenvalues of [[2,1],[1,2]].", "1 and 3"),
        ("Sum_{n=0}^{inf} 1/2^n.", "2"),
        ("Number of divisors of 360.", "24"),
        ("Solve e^{2x} - 3*e^x + 2 = 0.", "0 or log(2)"),
        ("d/dx of ln(x^2+1).", "2*x/(x**2+1)"),
        ("Sum 1^2 + 2^2 + ... + 10^2.", "385"),
        ("Solve x^4 - 5*x^2 + 4 = 0.", "1 or 2"),
        ("Critical points of f(x)=x^3-3*x.", "1 or -1"),
        ("Rank of [[1,2,3],[4,5,6],[7,8,9]].", "2"),
        ("How many primes between 1 and 50?", "15"),
        ("P(exactly 5 heads in 10 coin flips).", "63/256"),
        ("Find integer solutions to x^2+y^2=25.", "3 and 4"),
    ],
}

LEVEL_LABELS = {
    "Level_1": "Level 1 (easy)",
    "Level_3": "Level 3 (medium)",
    "Level_5": "Level 5 (hard)",
}
LEVEL_COLS = {"Level_1": "#2166AC", "Level_3": "#F4A582", "Level_5": "#D6604D"}
MODEL_COLS = {"phi35": "#1B7837", "qwen3b": "#762A83", "gemma2": "#E08214"}
MODEL_LABELS = {
    "phi35":  "Phi-3.5-mini (3.8B)",
    "qwen3b": "Qwen2.5-3B (3.0B)",
    "gemma2": "Gemma-2-2B (2.0B)",
}

# ── 2. CHECKPOINT I/O ─────────────────────────────────────────────────────────

CKPT_PATH = OUT / "checkpoint.json"

def load_checkpoint():
    if CKPT_PATH.exists():
        with open(CKPT_PATH) as f:
            return json.load(f)
    return {}

def save_checkpoint(ckpt):
    tmp = CKPT_PATH.with_suffix(".tmp")
    with open(tmp, "w") as f:
        json.dump(ckpt, f, indent=2, default=str)
    tmp.replace(CKPT_PATH)

def elapsed_min():
    return (time.time() - START_TIME) / 60

def time_ok():
    return elapsed_min() < SESSION_LIMIT_MIN

# ── 3. ANSWER CHECKING ────────────────────────────────────────────────────────

_SP_TRANSFORMS = standard_transformations + (implicit_multiplication_application,)

def _clean_latex(s):
    s = s.lower()
    s = re.sub(r"\\frac\s*{([^}]+)}\s*{([^}]+)}", r"(\1)/(\2)", s)
    s = re.sub(r"\\sqrt\s*{([^}]+)}", r"sqrt(\1)", s)
    for old, new in [
        (r"\\pi","pi"), (r"\\ln","log"), (r"\\log","log"),
        (r"\\sin","sin"), (r"\\cos","cos"), (r"\\cdot","*"),
        (r"\\times","*"), (r"\\infty","oo"),
    ]:
        s = re.sub(old, new, s)
    s = re.sub(r"[{}$\\]", " ", s)
    return s.strip()

def _sympy_eq(a, b):
    try:
        ea = parse_expr(a, transformations=_SP_TRANSFORMS)
        eb = parse_expr(b, transformations=_SP_TRANSFORMS)
        d  = sp.simplify(ea - eb)
        return d == 0 or abs(float(d.evalf())) < 1e-6
    except Exception:
        return False

def _extract_answer(text):
    for marker in ["ANSWER:", "Answer:", "= ", "equals "]:
        idx = text.rfind(marker)
        if idx >= 0:
            return text[idx + len(marker):].strip().split("\n")[0]
    return text.split(".")[-1].strip()

def answer_is_correct(generated, expected):
    gen_region = _extract_answer(generated)
    gen_clean  = _clean_latex(gen_region).replace(" ", "")
    parts = [p.strip() for p in expected.split("or") if p.strip()]
    for part in parts:
        part_clean = _clean_latex(part).replace(" ", "")
        if _sympy_eq(gen_clean, part_clean):
            return True
        if part_clean and part_clean in gen_clean:
            return True
    try:
        exp_num = float(sp.sympify(expected).evalf())
        nums = re.findall(r"-?\d+\.?\d*", gen_region)
        for n in nums:
            if abs(float(n) - exp_num) < 1e-4 * (abs(exp_num) + 1):
                return True
    except Exception:
        pass
    return False

# ── 4. g-MODEL FIT ────────────────────────────────────────────────────────────

def g_raw(T, Delta, sigma_r2, k):
    V = np.maximum(k * np.asarray(T, dtype=float), 0.0)
    s = np.maximum(sigma_r2 + V, 1e-12)
    return (1.0 / np.sqrt(2.0 * np.pi * s)) * np.exp(-Delta**2 / (2.0 * s))

def g_scaled(T, Delta, sigma_r2, k, scale, offset):
    return scale * g_raw(T, Delta, sigma_r2, k) + offset

def fit_g(T_arr, A_arr, n_boot=400):
    T = np.asarray(T_arr, dtype=float)
    A = np.asarray(A_arr, dtype=float)
    best_r2, best_p = -np.inf, None
    for Di in [0.3, 0.5, 0.8, 1.2, 2.0]:
        for ki in [1e-5, 5e-5, 1e-4, 5e-4]:
            try:
                T_pk = T[np.argmax(A)]
                gp   = g_raw(np.array([T_pk]), Di, 0.05, ki)[0]
                sc0  = max(A) / max(gp, 1e-10)
                popt, _ = curve_fit(
                    g_scaled, T, A,
                    p0=[Di, 0.05, ki, sc0, 0.0],
                    bounds=([0.01,1e-4,1e-9, 1.0,-50.0],
                            [5.00,5.00,1.00,1e6, 50.0]),
                    maxfev=20000, method="trf",
                )
                ss_r = np.sum((A - g_scaled(T, *popt))**2)
                ss_t = np.sum((A - A.mean())**2)
                r2   = 1.0 - ss_r / ss_t if ss_t > 0 else -np.inf
                if r2 > best_r2:
                    best_r2, best_p = r2, popt
            except Exception:
                continue
    if best_p is None:
        return {"error": "fit_failed"}
    Delta, sr2, k, sc, off = best_p
    T_fine = np.linspace(T.min(), T.max() * 2.5, 8000)
    A_fine = g_scaled(T_fine, *best_p)
    T_star = float(T_fine[np.argmax(A_fine)])
    peak   = float(np.max(A_fine))
    n   = len(T)
    rss = float(np.sum((A - g_scaled(T, *best_p))**2))
    kp  = len(best_p)
    aic = n * np.log(rss / n + 1e-15) + 2 * kp
    bic = n * np.log(rss / n + 1e-15) + kp * np.log(n)
    # bootstrap CI on T*
    ci  = {}
    bts = []
    rng = np.random.default_rng(42)
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        Tb, Ab = T[idx], A[idx]
        if len(np.unique(Tb)) < 3:
            continue
        try:
            pb, _ = curve_fit(
                g_scaled, Tb, Ab, p0=best_p,
                bounds=([0.01,1e-4,1e-9, 1.0,-50.0],
                        [5.00,5.00,1.00,1e6, 50.0]),
                maxfev=5000, method="trf",
            )
            bts.append(pb)
        except Exception:
            continue
    if len(bts) >= 40:
        ba = np.array(bts)
        for i, nm in enumerate(["Delta","sigma_r2","k","scale","offset"]):
            ci[nm] = [float(np.percentile(ba[:,i], 2.5)),
                      float(np.percentile(ba[:,i], 97.5))]
        bt_s = []
        for pb in ba:
            Tf2 = np.linspace(T.min(), T.max() * 2.5, 2000)
            bt_s.append(float(Tf2[np.argmax(g_scaled(Tf2, *pb))]))
        ci["T_star"] = [float(np.percentile(bt_s, 2.5)),
                        float(np.percentile(bt_s, 97.5))]
    return {
        "Delta": float(Delta), "sigma_r2": float(sr2),
        "k": float(k), "scale": float(sc), "offset": float(off),
        "R2": float(best_r2), "T_star": T_star, "peak_acc": peak,
        "AIC": float(aic), "BIC": float(bic),
        "CI": ci, "n_data": int(n),
    }

# ── 5. MODEL HELPERS ──────────────────────────────────────────────────────────

def load_model_8bit(model_id):
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    bnb = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb,
        device_map="auto", trust_remote_code=False,
    )
    model.eval()
    return model, tok

def release(model):
    del model; gc.collect(); torch.cuda.empty_cache()

def build_prompt(tok, problem, sys_prompt, has_system):
    """Return a tokenised prompt using the model's chat template."""
    if has_system:
        messages = [
            {"role": "system",    "content": sys_prompt},
            {"role": "user",      "content": problem},
        ]
    else:
        # Gemma-2 and similar: fold system prompt into user turn
        messages = [{"role": "user", "content": f"{sys_prompt}\n\n{problem}"}]
    try:
        text = tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        # Fallback for models without a registered chat template
        if has_system:
            text = f"System: {sys_prompt}\nUser: {problem}\nAssistant:"
        else:
            text = f"{sys_prompt}\n{problem}\nAnswer:"
    return tok(text, return_tensors="pt").to(DEVICE)

# ── 6. SWEEP FUNCTION ─────────────────────────────────────────────────────────

def run_sweep(model, tok, problems, budgets, model_key, n_samples=N_SAMPLES):
    """
    Returns list[dict] with one entry per budget:
      {"budget": int, "accuracy": float, "mean_ent": float, "V_between": float}
    """
    has_sys = MODEL_HAS_SYSTEM.get(model_key, True)
    results = []
    for budget in budgets:
        all_correct  = []
        all_ent      = []
        all_V_between = []
        for problem, expected in problems:
            inputs = build_prompt(tok, problem, SYS_PROMPT, has_sys)
            in_len = inputs["input_ids"].shape[1]
            run_correct = []
            run_ents    = []
            for _ in range(n_samples):
                with torch.no_grad():
                    out = model.generate(
                        **inputs,
                        max_new_tokens=budget,
                        do_sample=True,
                        temperature=0.7,
                        top_p=0.9,
                        output_scores=True,
                        return_dict_in_generate=True,
                        pad_token_id=tok.eos_token_id,
                    )
                gen_ids  = out.sequences[0][in_len:]
                gen_text = tok.decode(gen_ids, skip_special_tokens=True)
                run_correct.append(int(answer_is_correct(gen_text, expected)))
                if out.scores:
                    ents = []
                    for sc in out.scores:
                        p = torch.softmax(sc[0], dim=-1).cpu().numpy()
                        p = p[p > 1e-12]
                        ents.append(float(-np.sum(p * np.log(p))))
                    run_ents.append(float(np.mean(ents)))
            all_correct.append(float(np.mean(run_correct)))
            if len(run_ents) >= 2:
                all_V_between.append(float(np.var(run_ents, ddof=1)))
            if run_ents:
                all_ent.append(float(np.mean(run_ents)))
        acc       = float(np.mean(all_correct))
        mean_ent  = float(np.mean(all_ent))      if all_ent      else float("nan")
        V_between = float(np.mean(all_V_between)) if all_V_between else float("nan")
        results.append({"budget": budget, "accuracy": acc,
                        "mean_ent": mean_ent, "V_between": V_between})
        print(f"      T={budget:5d}  acc={acc:.3f}  H={mean_ent:.4f}")
    return results

# ── 7. MAIN EXPERIMENT LOOP ───────────────────────────────────────────────────

ckpt = load_checkpoint()
print(f"\n[Checkpoint] loaded {len(ckpt)} completed (model, level) pairs.")

for model_key, model_id in MODELS.items():
    if not time_ok():
        print(f"\n[WARNING] Approaching session limit at {elapsed_min():.1f} min — stopping.")
        break

    # check if ALL levels done for this model
    levels_needed = [lv for lv in MATH_PROBLEMS
                     if f"{model_key}_{lv}" not in ckpt]
    if not levels_needed:
        print(f"\n[SKIP] {model_key} — all levels already in checkpoint.")
        continue

    print(f"\n{'='*66}")
    print(f"  Loading {MODEL_LABELS[model_key]}  ({model_id})  [{elapsed_min():.1f} min]")
    print(f"  Levels remaining: {levels_needed}")
    print(f"{'='*66}")

    try:
        model, tok = load_model_8bit(model_id)
        print(f"  Model loaded  [{elapsed_min():.1f} min]")
    except Exception as e:
        print(f"  ERROR loading {model_key}: {e}")
        ckpt[f"{model_key}_load_error"] = str(e)
        save_checkpoint(ckpt)
        continue

    for level in levels_needed:
        ck_key = f"{model_key}_{level}"
        if ck_key in ckpt:
            print(f"  [skip] {ck_key} already done.")
            continue
        if not time_ok():
            print(f"  [TIME] stopping before {ck_key}")
            break

        problems = MATH_PROBLEMS[level][:N_PROBS]
        print(f"\n  [{model_key}][{level}]  "
              f"{len(problems)} problems × {len(BUDGETS)} budgets × {N_SAMPLES} samples")

        sweep = run_sweep(model, tok, problems, BUDGETS, model_key)

        T_arr = np.array([r["budget"]       for r in sweep])
        A_arr = np.array([r["accuracy"]*100 for r in sweep])
        fit   = fit_g(T_arr, A_arr, n_boot=400)

        print(f"  -> R²={fit.get('R2',0):.4f}  "
              f"T*={fit.get('T_star',0):.0f}  "
              f"peak={fit.get('peak_acc',0):.1f}%")

        ckpt[ck_key] = {"sweep": sweep, "fit": fit,
                        "model_id": model_id, "level": level,
                        "elapsed_min": elapsed_min()}
        save_checkpoint(ckpt)
        print(f"  [ckpt] {ck_key} saved  [{elapsed_min():.1f} min]")

    release(model)
    print(f"  [released] {model_key}  [{elapsed_min():.1f} min]")

# ── 8. ASSEMBLE FINAL RESULTS ─────────────────────────────────────────────────

RESULTS = {"models": {}, "fits": {}, "P1_per_model": {}}

for model_key in MODELS:
    RESULTS["models"][model_key] = {}
    RESULTS["fits"][model_key]   = {}
    for level in MATH_PROBLEMS:
        ck_key = f"{model_key}_{level}"
        if ck_key in ckpt and "sweep" in ckpt[ck_key]:
            RESULTS["models"][model_key][level] = ckpt[ck_key]["sweep"]
            RESULTS["fits"][model_key][level]   = ckpt[ck_key]["fit"]

# P1 test per model: T*(L1) < T*(L3) < T*(L5)?
for model_key in MODELS:
    fits = RESULTS["fits"].get(model_key, {})
    t_stars = {lv: fits[lv]["T_star"] for lv in ["Level_1","Level_3","Level_5"]
               if lv in fits and "T_star" in fits[lv]}
    if len(t_stars) == 3:
        ok = (t_stars["Level_1"] < t_stars["Level_3"] < t_stars["Level_5"])
        RESULTS["P1_per_model"][model_key] = {
            **t_stars, "monotonic": ok,
            "ratio": t_stars["Level_5"] / max(t_stars["Level_1"], 1),
        }

out_json = OUT / "multimodel_results.json"
with open(out_json, "w") as f:
    json.dump(RESULTS, f, indent=2, default=str)
print(f"\n[json] multimodel_results.json  [{elapsed_min():.1f} min]")

# ── 9. FIGURES ────────────────────────────────────────────────────────────────

plt.rcParams.update({
    "font.family":"serif","font.size":11,"axes.titlesize":12,
    "axes.labelsize":11,"legend.fontsize":9,"savefig.dpi":300,
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.alpha":0.25,
})

def savefig(name, fig=None):
    fig = fig or plt.gcf()
    for ext in ("pdf","png"):
        fig.savefig(OUT / f"{name}.{ext}", bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"  [fig] {name}")

# ── Figure A1: one panel per model, 3 curves per level ──────────────────────
n_models_done = sum(1 for mk in MODELS if any(
    f"{mk}_{lv}" in ckpt and "sweep" in ckpt[f"{mk}_{lv}"]
    for lv in MATH_PROBLEMS))

if n_models_done > 0:
    fig, axes = plt.subplots(1, max(n_models_done, 1),
                             figsize=(5.5 * n_models_done, 4.8),
                             squeeze=False)
    ax_idx = 0
    for model_key, model_id in MODELS.items():
        model_data = RESULTS["models"].get(model_key, {})
        model_fits = RESULTS["fits"].get(model_key, {})
        if not model_data:
            continue
        ax = axes[0][ax_idx]; ax_idx += 1
        for level in ["Level_1", "Level_3", "Level_5"]:
            if level not in model_data:
                continue
            col  = LEVEL_COLS[level]
            data = model_data[level]
            T_d  = np.array([r["budget"]       for r in data])
            A_d  = np.array([r["accuracy"]*100 for r in data])
            ax.scatter(T_d/1000, A_d, color=col, s=40, zorder=6)
            ax.plot(T_d/1000, A_d, "o-", color=col, lw=1.3, alpha=0.4)
            fit = model_fits.get(level, {})
            if "Delta" in fit:
                popt = [fit["Delta"], fit["sigma_r2"], fit["k"],
                        fit["scale"], fit["offset"]]
                Tf = np.linspace(T_d.min(), T_d.max()*1.4, 2000)
                ax.plot(Tf/1000, g_scaled(Tf, *popt), color=col, lw=2.0,
                        label=f"{LEVEL_LABELS[level]}  T*={fit['T_star']:.0f}")
                ax.axvline(fit["T_star"]/1000, color=col, lw=0.8, ls=":", alpha=0.7)
        ax.set_xlabel("Budget $T$ (k tokens)")
        ax.set_ylabel("Accuracy (%)")
        ax.set_title(MODEL_LABELS[model_key])
        ax.legend(fontsize=8, loc="upper left")
    fig.suptitle("Experiment A: Overthinking curves — multi-model validation\n"
                 "(8 problems/level, 2 runs, 8-bit quantisation)",
                 fontsize=11, y=1.02)
    plt.tight_layout()
    savefig("figure_multimodel", fig)

# ── Figure A2: T* comparison table bar chart ─────────────────────────────────
p1_data = RESULTS.get("P1_per_model", {})
done_models = [mk for mk in MODELS if mk in p1_data]
if done_models:
    levels_plot = ["Level_1","Level_3","Level_5"]
    x = np.arange(len(levels_plot))
    width = 0.25
    fig, ax = plt.subplots(figsize=(9, 4.8))
    for i, mk in enumerate(done_models):
        t_vals = [p1_data[mk].get(lv, 0) for lv in levels_plot]
        bars   = ax.bar(x + i*width, t_vals, width,
                        color=MODEL_COLS[mk], label=MODEL_LABELS[mk],
                        alpha=0.82, zorder=3)
        for bar, v in zip(bars, t_vals):
            if v:
                ax.text(bar.get_x()+bar.get_width()/2, v + 30,
                        f"{v:.0f}", ha="center", fontsize=7, rotation=45)
    ax.set_xticks(x + width)
    ax.set_xticklabels([LEVEL_LABELS[lv] for lv in levels_plot])
    ax.set_ylabel("Optimal budget $T^*$ (tokens)")
    ax.set_title("Experiment A: T* by model and difficulty level\n"
                 "P1 prediction: T* increases with difficulty (for all models)")
    ax.legend(fontsize=9)
    plt.tight_layout()
    savefig("figure_multimodel_tstar", fig)

# ── 10. LaTeX TABLE ───────────────────────────────────────────────────────────

def tex(v, fmt=".3f"):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return "—"
    return format(v, fmt)

lines = [
    "% AUTO-GENERATED by part3_multimodel.py",
    r"\begin{table}[t]",
    r"\centering\small",
    r"\caption{Experiment A: g-model fits per model and difficulty level. "
    r"$T^*$ = fitted overthinking threshold (tokens); "
    r"Peak = fitted peak accuracy (\%); "
    r"P1 = T* monotone-increasing with difficulty.}",
    r"\label{tab:multimodel}",
    r"\begin{tabular}{llccccc}",
    r"\toprule",
    r"Model & Level & $T^*$ & 95\% CI & Peak (\%) & $R^2$ & P1 \\",
    r"\midrule",
]
for model_key in MODELS:
    model_fits = RESULTS["fits"].get(model_key, {})
    p1_info    = RESULTS["P1_per_model"].get(model_key, {})
    p1_str     = "✓" if p1_info.get("monotonic") else "✗"
    for li, level in enumerate(["Level_1","Level_3","Level_5"]):
        fit  = model_fits.get(level, {})
        ci   = fit.get("CI", {})
        t_ci = ci.get("T_star", [float("nan"), float("nan")])
        mlbl = MODEL_LABELS[model_key] if li == 0 else ""
        p1c  = p1_str if li == 2 else ""
        if "T_star" not in fit:
            lines.append(rf"{mlbl} & {level.replace('_',' ')} & \multicolumn{{4}}{{c}}{{not completed}} & {p1c} \\")
        else:
            lines.append(
                rf"{mlbl} & {level.replace('_',' ')} & "
                rf"${tex(fit.get('T_star'),'.0f')}$ & "
                rf"$[{tex(t_ci[0],'.0f')},{tex(t_ci[1],'.0f')}]$ & "
                rf"${tex(fit.get('peak_acc'),'.1f')}$ & "
                rf"${tex(fit.get('R2'))}$ & {p1c} \\\\"
            )
    lines.append(r"\midrule")
lines[-1] = r"\bottomrule"
lines += [r"\end{tabular}", r"\end{table}"]
tex_path = OUT / "table_multimodel.tex"
tex_path.write_text("\n".join(lines), encoding="utf-8")
print(f"  [tex] table_multimodel.tex")

# ── 11. SUMMARY ───────────────────────────────────────────────────────────────

print("\n" + "="*66)
print("  EXPERIMENT A SUMMARY")
print("="*66)
for model_key in MODELS:
    p1 = RESULTS["P1_per_model"].get(model_key, {})
    if p1:
        print(f"  {MODEL_LABELS[model_key]}")
        for lv in ["Level_1","Level_3","Level_5"]:
            print(f"    {lv}: T*={p1.get(lv,0):.0f}")
        print(f"    P1 supported: {p1.get('monotonic','?')}  "
              f"ratio={p1.get('ratio',0):.1f}x")
    else:
        print(f"  {MODEL_LABELS[model_key]}: incomplete")
print(f"\n  Total elapsed: {elapsed_min():.1f} min")
print(f"  Outputs: {OUT}")
for fp in sorted(OUT.iterdir()):
    print(f"    {fp.name:48s}  {fp.stat().st_size/1024:7.1f} KB")

---
## Experiment B — P6: T* Decreases with Model Size

**Prediction P6 tested:** dT*/dN < 0 — larger models overthink at lower token budgets.  
Models: Qwen2.5-1.5B, 3B, 7B-Instruct (same architecture family, only N varies).


In [ ]:
import os, sys, json, time, math, warnings, gc, re
from pathlib import Path

warnings.filterwarnings("ignore")
START_TIME = time.time()

import subprocess
for pkg in ["torch", "transformers", "accelerate", "bitsandbytes",
            "scipy", "scikit-learn", "sympy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                   capture_output=True)

import numpy as np
import scipy.stats as stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import torch
import sympy as sp
from sympy.parsing.sympy_parser import (
    parse_expr, standard_transformations, implicit_multiplication_application,
)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

torch.manual_seed(42); np.random.seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Setup] device={DEVICE}  GPUs={torch.cuda.device_count()}")
if DEVICE == "cuda":
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB")

OUT = Path("/kaggle/working/results_part4")
OUT.mkdir(parents=True, exist_ok=True)

# ── 1. CONSTANTS ──────────────────────────────────────────────────────────────

BUDGETS   = [200, 500, 1000, 2000, 4000, 8000]
N_SAMPLES = 3
N_PROBS   = 10
SESSION_LIMIT_MIN = 510

# Model N values in millions (for plotting)
MODELS = {
    "qwen_1b5": ("Qwen/Qwen2.5-1.5B-Instruct", 1500),
    "qwen_3b":  ("Qwen/Qwen2.5-3B-Instruct",   3000),
    "qwen_7b":  ("Qwen/Qwen2.5-7B-Instruct",   7000),
}
MODEL_LABELS = {
    "qwen_1b5": "Qwen2.5-1.5B",
    "qwen_3b":  "Qwen2.5-3B",
    "qwen_7b":  "Qwen2.5-7B",
}
MODEL_COLS   = {
    "qwen_1b5": "#1F78B4",
    "qwen_3b":  "#33A02C",
    "qwen_7b":  "#E31A1C",
}
LEVEL_COLS   = {"Level_1":"#2166AC","Level_3":"#F4A582","Level_5":"#D6604D"}

SYS_PROMPT = (
    "You are a precise math assistant. "
    "Think step by step. "
    "Write your final numerical answer after the word ANSWER: on its own line."
)

MATH_PROBLEMS = {
    "Level_1": [
        ("Compute $2^3 + 3^2$.", "17"),
        ("What is $15\\%$ of $200$?", "30"),
        ("If $x + 5 = 12$, find $x$.", "7"),
        ("What is $\\sqrt{49}$?", "7"),
        ("Calculate $3!$.", "6"),
        ("What is $4^2 - 3^2$?", "7"),
        ("Find $x$: $2x = 18$.", "9"),
        ("Compute $7 \\times 8$.", "56"),
        ("What is $2^5$?", "32"),
        ("Find the mean of $4, 6, 8, 10$.", "7"),
    ],
    "Level_3": [
        ("Solve $x^2 - 5x + 6 = 0$.", "2 or 3"),
        ("A circle has radius 7. Area in terms of pi.", "49*pi"),
        ("If $f(x)=2x^2-3x+1$, find $f(3)$.", "10"),
        ("Solve: $\\log_2 8 = x$.", "3"),
        ("Distance between $(0,0)$ and $(3,4)$.", "5"),
        ("What is $C(6,2)$?", "15"),
        ("Find the 10th term of $3,7,11,\\dots$", "39"),
        ("Solve $|x-3|=5$.", "8 or -2"),
        ("Solve $2^x=32$.", "5"),
        ("Sum of first 10 natural numbers.", "55"),
    ],
    "Level_5": [
        ("Compute sum_{k=1}^{inf} 1/(k(k+1)).", "1"),
        ("Evaluate integral_0^pi sin^2(x) dx.", "pi/2"),
        ("Limit as x->0 of sin(3x)/x.", "3"),
        ("Eigenvalues of [[2,1],[1,2]].", "1 and 3"),
        ("Sum_{n=0}^{inf} 1/2^n.", "2"),
        ("Number of divisors of 360.", "24"),
        ("Solve e^{2x} - 3*e^x + 2 = 0.", "0 or log(2)"),
        ("d/dx of ln(x^2+1).", "2*x/(x**2+1)"),
        ("Sum 1^2 + 2^2 + ... + 10^2.", "385"),
        ("Solve x^4 - 5*x^2 + 4 = 0.", "1 or 2"),
    ],
}

# ── 2. CHECKPOINT I/O ─────────────────────────────────────────────────────────

CKPT_PATH = OUT / "checkpoint.json"

def load_checkpoint():
    if CKPT_PATH.exists():
        with open(CKPT_PATH) as f:
            return json.load(f)
    return {}

def save_checkpoint(ckpt):
    tmp = CKPT_PATH.with_suffix(".tmp")
    with open(tmp, "w") as f:
        json.dump(ckpt, f, indent=2, default=str)
    tmp.replace(CKPT_PATH)

def elapsed_min():
    return (time.time() - START_TIME) / 60

def time_ok():
    return elapsed_min() < SESSION_LIMIT_MIN

# ── 3. ANSWER CHECKING ────────────────────────────────────────────────────────

_SP_TRANSFORMS = standard_transformations + (implicit_multiplication_application,)

def _clean_latex(s):
    s = s.lower()
    s = re.sub(r"\\frac\s*{([^}]+)}\s*{([^}]+)}", r"(\1)/(\2)", s)
    s = re.sub(r"\\sqrt\s*{([^}]+)}", r"sqrt(\1)", s)
    for old, new in [
        (r"\\pi","pi"),(r"\\ln","log"),(r"\\log","log"),
        (r"\\sin","sin"),(r"\\cos","cos"),(r"\\cdot","*"),
        (r"\\times","*"),(r"\\infty","oo"),
    ]:
        s = re.sub(old, new, s)
    s = re.sub(r"[{}$\\]", " ", s)
    return s.strip()

def _sympy_eq(a, b):
    try:
        ea = parse_expr(a, transformations=_SP_TRANSFORMS)
        eb = parse_expr(b, transformations=_SP_TRANSFORMS)
        d  = sp.simplify(ea - eb)
        return d == 0 or abs(float(d.evalf())) < 1e-6
    except Exception:
        return False

def _extract_answer(text):
    for marker in ["ANSWER:", "Answer:", "= ", "equals "]:
        idx = text.rfind(marker)
        if idx >= 0:
            return text[idx + len(marker):].strip().split("\n")[0]
    return text.split(".")[-1].strip()

def answer_is_correct(generated, expected):
    gen_region = _extract_answer(generated)
    gen_clean  = _clean_latex(gen_region).replace(" ", "")
    parts = [p.strip() for p in expected.split("or") if p.strip()]
    for part in parts:
        part_clean = _clean_latex(part).replace(" ", "")
        if _sympy_eq(gen_clean, part_clean):
            return True
        if part_clean and part_clean in gen_clean:
            return True
    try:
        exp_num = float(sp.sympify(expected).evalf())
        nums = re.findall(r"-?\d+\.?\d*", gen_region)
        for n in nums:
            if abs(float(n) - exp_num) < 1e-4 * (abs(exp_num) + 1):
                return True
    except Exception:
        pass
    return False

# ── 4. g-MODEL FIT ────────────────────────────────────────────────────────────

def g_raw(T, Delta, sigma_r2, k):
    V = np.maximum(k * np.asarray(T, dtype=float), 0.0)
    s = np.maximum(sigma_r2 + V, 1e-12)
    return (1.0 / np.sqrt(2.0 * np.pi * s)) * np.exp(-Delta**2 / (2.0 * s))

def g_scaled(T, Delta, sigma_r2, k, scale, offset):
    return scale * g_raw(T, Delta, sigma_r2, k) + offset

def fit_g(T_arr, A_arr, n_boot=300):
    T = np.asarray(T_arr, dtype=float)
    A = np.asarray(A_arr, dtype=float)
    best_r2, best_p = -np.inf, None
    for Di in [0.3, 0.5, 0.8, 1.2, 2.0]:
        for ki in [1e-5, 5e-5, 1e-4, 5e-4]:
            try:
                T_pk = T[np.argmax(A)]
                gp   = g_raw(np.array([T_pk]), Di, 0.05, ki)[0]
                sc0  = max(A) / max(gp, 1e-10)
                popt, _ = curve_fit(
                    g_scaled, T, A,
                    p0=[Di, 0.05, ki, sc0, 0.0],
                    bounds=([0.01,1e-4,1e-9, 1.0,-50.0],
                            [5.00,5.00,1.00,1e6, 50.0]),
                    maxfev=20000, method="trf",
                )
                ss_r = np.sum((A - g_scaled(T, *popt))**2)
                ss_t = np.sum((A - A.mean())**2)
                r2   = 1.0 - ss_r / ss_t if ss_t > 0 else -np.inf
                if r2 > best_r2:
                    best_r2, best_p = r2, popt
            except Exception:
                continue
    if best_p is None:
        return {"error": "fit_failed"}
    Delta, sr2, k, sc, off = best_p
    T_fine = np.linspace(T.min(), T.max() * 2.5, 8000)
    A_fine = g_scaled(T_fine, *best_p)
    T_star = float(T_fine[np.argmax(A_fine)])
    peak   = float(np.max(A_fine))
    n  = len(T)
    rss= float(np.sum((A - g_scaled(T, *best_p))**2))
    kp = len(best_p)
    # bootstrap CI on T*
    ci  = {}
    bts = []
    rng = np.random.default_rng(42)
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        Tb, Ab = T[idx], A[idx]
        if len(np.unique(Tb)) < 3:
            continue
        try:
            pb, _ = curve_fit(
                g_scaled, Tb, Ab, p0=best_p,
                bounds=([0.01,1e-4,1e-9, 1.0,-50.0],
                        [5.00,5.00,1.00,1e6, 50.0]),
                maxfev=5000, method="trf",
            )
            bts.append(pb)
        except Exception:
            continue
    if len(bts) >= 40:
        ba = np.array(bts)
        bt_s = []
        for pb in ba:
            Tf2 = np.linspace(T.min(), T.max() * 2.5, 2000)
            bt_s.append(float(Tf2[np.argmax(g_scaled(Tf2, *pb))]))
        ci["T_star"] = [float(np.percentile(bt_s, 2.5)),
                        float(np.percentile(bt_s, 97.5))]
    return {
        "Delta":float(Delta),"sigma_r2":float(sr2),"k":float(k),
        "scale":float(sc),"offset":float(off),
        "R2":float(best_r2),"T_star":T_star,"peak_acc":peak,
        "CI":ci,"n_data":int(n),
    }

# ── 5. MODEL / SWEEP HELPERS ──────────────────────────────────────────────────

def load_model_8bit(model_id):
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    bnb = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb,
        device_map="auto", trust_remote_code=False,
    )
    model.eval()
    return model, tok

def release(model):
    del model; gc.collect(); torch.cuda.empty_cache()

def build_prompt(tok, problem, sys_prompt):
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user",   "content": problem},
    ]
    try:
        text = tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        text = f"Instruct: {sys_prompt}\n{problem}\nOutput:"
    return tok(text, return_tensors="pt").to(DEVICE)

def run_sweep(model, tok, problems, budgets, n_samples=N_SAMPLES):
    results = []
    for budget in budgets:
        all_correct = []
        for problem, expected in problems:
            inputs = build_prompt(tok, problem, SYS_PROMPT)
            in_len = inputs["input_ids"].shape[1]
            run_correct = []
            for _ in range(n_samples):
                with torch.no_grad():
                    out = model.generate(
                        **inputs,
                        max_new_tokens=budget,
                        do_sample=True, temperature=0.7, top_p=0.9,
                        pad_token_id=tok.eos_token_id,
                    )
                sequences = out.sequences if hasattr(out, "sequences") else out
                gen_ids  = sequences[0][in_len:]
                gen_text = tok.decode(gen_ids, skip_special_tokens=True)
                run_correct.append(int(answer_is_correct(gen_text, expected)))
            all_correct.append(float(np.mean(run_correct)))
        acc = float(np.mean(all_correct))
        results.append({"budget": budget, "accuracy": acc})
        print(f"      T={budget:5d}  acc={acc:.3f}")
    return results

# ── 6. MAIN EXPERIMENT LOOP ───────────────────────────────────────────────────

ckpt = load_checkpoint()
print(f"\n[Checkpoint] {len(ckpt)} (model,level) pairs already done.")

for model_key, (model_id, param_M) in MODELS.items():
    if not time_ok():
        print(f"\n[TIME] session limit approaching — stopping.")
        break
    levels_needed = [lv for lv in MATH_PROBLEMS
                     if f"{model_key}_{lv}" not in ckpt]
    if not levels_needed:
        print(f"\n[SKIP] {model_key} — all levels done.")
        continue
    print(f"\n{'='*66}")
    print(f"  Loading {MODEL_LABELS[model_key]}  ({param_M}M params)  "
          f"[{elapsed_min():.1f} min]")
    print(f"  Levels remaining: {levels_needed}")
    print(f"{'='*66}")
    try:
        model, tok = load_model_8bit(model_id)
        print(f"  Model loaded  [{elapsed_min():.1f} min]")
    except Exception as e:
        print(f"  ERROR loading {model_key}: {e}")
        ckpt[f"{model_key}_load_error"] = str(e)
        save_checkpoint(ckpt)
        continue
    for level in levels_needed:
        ck_key = f"{model_key}_{level}"
        if not time_ok():
            print(f"  [TIME] stopping before {ck_key}")
            break
        problems = MATH_PROBLEMS[level][:N_PROBS]
        print(f"\n  [{model_key}][{level}]  "
              f"{len(problems)} probs × {len(BUDGETS)} budgets × {N_SAMPLES} runs")
        sweep = run_sweep(model, tok, problems, BUDGETS)
        T_arr = np.array([r["budget"]       for r in sweep])
        A_arr = np.array([r["accuracy"]*100 for r in sweep])
        fit   = fit_g(T_arr, A_arr)
        print(f"  -> R²={fit.get('R2',0):.4f}  T*={fit.get('T_star',0):.0f}")
        ckpt[ck_key] = {"sweep": sweep, "fit": fit, "param_M": param_M,
                        "model_id": model_id, "elapsed_min": elapsed_min()}
        save_checkpoint(ckpt)
        print(f"  [ckpt] {ck_key} saved  [{elapsed_min():.1f} min]")
    release(model)
    print(f"  [released] {model_key}  [{elapsed_min():.1f} min]")

# ── 7. ASSEMBLE RESULTS ───────────────────────────────────────────────────────

RESULTS = {"curves": {}, "fits": {}, "P6_tests": {}}

for model_key, (_, param_M) in MODELS.items():
    RESULTS["curves"][model_key] = {}
    RESULTS["fits"][model_key]   = {}
    for level in MATH_PROBLEMS:
        ck_key = f"{model_key}_{level}"
        if ck_key in ckpt and "sweep" in ckpt[ck_key]:
            RESULTS["curves"][model_key][level] = ckpt[ck_key]["sweep"]
            RESULTS["fits"][model_key][level]   = ckpt[ck_key]["fit"]

# P6 test per level: T* decreases with N?
model_keys = list(MODELS.keys())
param_Ms   = [MODELS[mk][1] for mk in model_keys]

for level in MATH_PROBLEMS:
    t_stars = {}
    for mk in model_keys:
        fit = RESULTS["fits"].get(mk, {}).get(level, {})
        if "T_star" in fit:
            t_stars[mk] = fit["T_star"]
    if len(t_stars) < 2:
        continue
    # Check monotone decrease with N (increasing param_M)
    ordered_t = [t_stars.get(mk, float("nan")) for mk in model_keys]
    valid = [v for v in ordered_t if not math.isnan(v)]
    monotone_dec = all(valid[i] >= valid[i+1] for i in range(len(valid)-1))
    RESULTS["P6_tests"][level] = {
        "T_stars": t_stars,
        "param_Ms": {mk: MODELS[mk][1] for mk in t_stars},
        "monotone_decreasing_with_N": monotone_dec,
    }

out_json = OUT / "p6_results.json"
with open(out_json, "w") as f:
    json.dump(RESULTS, f, indent=2, default=str)
print(f"\n[json] p6_results.json  [{elapsed_min():.1f} min]")

# ── 8. FIGURES ────────────────────────────────────────────────────────────────

plt.rcParams.update({
    "font.family":"serif","font.size":11,"axes.titlesize":12,
    "axes.labelsize":11,"legend.fontsize":9,"savefig.dpi":300,
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.alpha":0.25,
})

def savefig(name, fig=None):
    fig = fig or plt.gcf()
    for ext in ("pdf","png"):
        fig.savefig(OUT / f"{name}.{ext}", bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"  [fig] {name}")

# ── Figure B1: Overthinking curves for Level_3 (all 3 models) ────────────────
level_plot = "Level_3"
has_data = any(
    level_plot in RESULTS["curves"].get(mk, {}) for mk in model_keys
)
if has_data:
    fig, ax = plt.subplots(figsize=(8, 4.8))
    for mk in model_keys:
        data = RESULTS["curves"].get(mk, {}).get(level_plot)
        if not data:
            continue
        col = MODEL_COLS[mk]
        T_d = np.array([r["budget"]       for r in data])
        A_d = np.array([r["accuracy"]*100 for r in data])
        ax.scatter(T_d/1000, A_d, color=col, s=50, zorder=6)
        ax.plot(T_d/1000, A_d, "o-", color=col, lw=1.3, alpha=0.4)
        fit = RESULTS["fits"].get(mk, {}).get(level_plot, {})
        if "Delta" in fit:
            popt = [fit["Delta"],fit["sigma_r2"],fit["k"],fit["scale"],fit["offset"]]
            Tf   = np.linspace(T_d.min(), T_d.max()*1.4, 2000)
            ax.plot(Tf/1000, g_scaled(Tf, *popt), color=col, lw=2.2,
                    label=f"{MODEL_LABELS[mk]}  T*={fit['T_star']:.0f}")
            ax.axvline(fit["T_star"]/1000, color=col, lw=0.9, ls=":", alpha=0.7)
            ci = fit.get("CI", {})
            if "T_star" in ci:
                ax.axvspan(ci["T_star"][0]/1000, ci["T_star"][1]/1000,
                           alpha=0.07, color=col)
    ax.set_xlabel("Budget $T$ (thousands of tokens)")
    ax.set_ylabel("Accuracy (%) — Level 3 (medium)")
    ax.set_title("Experiment B: P6 model-size scaling\n"
                 r"Prediction: $T^*$ decreases as $N$ increases")
    ax.legend(fontsize=9)
    savefig("figure_p6_scaling", fig)

# ── Figure B2: T* vs log(N) scatter for all levels ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=False)
for ax_i, level in enumerate(["Level_1","Level_3","Level_5"]):
    ax = axes[ax_i]
    p6t = RESULTS["P6_tests"].get(level, {})
    t_stars = p6t.get("T_stars", {})
    xs, ys, cs = [], [], []
    for mk in model_keys:
        if mk in t_stars:
            xs.append(np.log10(MODELS[mk][1]))
            ys.append(t_stars[mk])
            cs.append(MODEL_COLS[mk])
    if xs:
        ax.scatter(xs, ys, c=cs, s=120, zorder=6)
        for mk, x, y in zip([m for m in model_keys if m in t_stars], xs, ys):
            ax.annotate(MODEL_LABELS[mk], (x, y),
                        textcoords="offset points", xytext=(5,5), fontsize=8)
        if len(xs) >= 2:
            m, b, r, p, _ = stats.linregress(xs, ys)
            xr = np.array([min(xs)-0.1, max(xs)+0.1])
            ax.plot(xr, m*xr+b, "k--", lw=1.5, alpha=0.5,
                    label=f"slope={m:.0f}, R²={r**2:.2f}")
            ax.legend(fontsize=8)
    ax.set_xlabel(r"$\log_{10}(N)$ (model params in M)")
    ax.set_ylabel("$T^*$ (tokens)")
    ax.set_title(f"{level.replace('_',' ')} — "
                 f"P6 {'✓' if p6t.get('monotone_decreasing_with_N') else '✗'}")
plt.suptitle("P6 Test: T* vs model size (Qwen2.5 family)\n"
             "P6 predicts: larger N → smaller T*", y=1.02, fontsize=11)
plt.tight_layout()
savefig("figure_p6_tstar", fig)

# ── 9. LaTeX TABLE ────────────────────────────────────────────────────────────

def tex(v, fmt=".3f"):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return "—"
    return format(v, fmt)

lines = [
    "% AUTO-GENERATED by part4_p6_scaling.py",
    r"\begin{table}[t]",
    r"\centering\small",
    r"\caption{Experiment B (P6): g-model fits for Qwen2.5 family. "
    r"P6 predicts $\partial T^*/\partial N < 0$: larger models reach the "
    r"overthinking threshold sooner. "
    r"{\checkmark} = monotone-decreasing T* with N.}",
    r"\label{tab:p6-scaling}",
    r"\begin{tabular}{llccc}",
    r"\toprule",
    r"Model ($N$) & Level & $T^*$ (tokens) & 95\% CI & P6 \\",
    r"\midrule",
]
for mk, (model_id, param_M) in MODELS.items():
    for li, level in enumerate(["Level_1","Level_3","Level_5"]):
        fit  = RESULTS["fits"].get(mk, {}).get(level, {})
        ci   = fit.get("CI", {})
        t_ci = ci.get("T_star", [float("nan"), float("nan")])
        p6t  = RESULTS["P6_tests"].get(level, {})
        p6c  = r"\checkmark" if p6t.get("monotone_decreasing_with_N") else r"\texttimes"
        mlbl = f"{MODEL_LABELS[mk]} ({param_M}M)" if li == 0 else ""
        p6col= p6c if li == 2 else ""
        if "T_star" not in fit:
            lines.append(rf"{mlbl} & {level.replace('_',' ')} & — & — & {p6col} \\")
        else:
            lines.append(
                rf"{mlbl} & {level.replace('_',' ')} & "
                rf"${tex(fit['T_star'],'.0f')}$ & "
                rf"$[{tex(t_ci[0],'.0f')},{tex(t_ci[1],'.0f')}]$ & "
                rf"{p6col} \\"
            )
    lines.append(r"\midrule")
lines[-1] = r"\bottomrule"
lines += [r"\end{tabular}", r"\end{table}"]
(OUT / "table_p6.tex").write_text("\n".join(lines), encoding="utf-8")
print("  [tex] table_p6.tex")

# ── 10. SUMMARY ───────────────────────────────────────────────────────────────

print("\n" + "="*66)
print("  EXPERIMENT B (P6) SUMMARY")
print("="*66)
for level in ["Level_1","Level_3","Level_5"]:
    p6t = RESULTS["P6_tests"].get(level, {})
    t_stars = p6t.get("T_stars", {})
    ok = p6t.get("monotone_decreasing_with_N","?")
    vals = "  ".join(
        f"{MODEL_LABELS[mk]}={t_stars[mk]:.0f}" for mk in model_keys if mk in t_stars
    )
    print(f"  {level}: {vals}  P6={'SUPPORTED' if ok else 'NOT SUPPORTED'}")
print(f"\n  Total elapsed: {elapsed_min():.1f} min")
print(f"  Outputs: {OUT}")
for fp in sorted(OUT.iterdir()):
    print(f"    {fp.name:48s}  {fp.stat().st_size/1024:7.1f} KB")

---
## Experiment C — Hurst Exponent / Variance Growth (P5)

**Prediction P5 tested:** V(T) ∝ T^{2H} with H ≈ 0.5 (random-walk variance growth).  
Both Phi-3.5-mini and Qwen2.5-3B tested for cross-model replication.


In [ ]:
import os, sys, json, time, math, warnings, gc, re
from pathlib import Path

warnings.filterwarnings("ignore")
START_TIME = time.time()

import subprocess
for pkg in ["torch", "transformers", "accelerate", "bitsandbytes", "scipy", "sympy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                   capture_output=True)

import numpy as np
import scipy.stats as stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import sympy as sp
from sympy.parsing.sympy_parser import (
    parse_expr, standard_transformations, implicit_multiplication_application,
)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

torch.manual_seed(42); np.random.seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Setup] device={DEVICE}  GPUs={torch.cuda.device_count()}")
if DEVICE == "cuda":
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB")

OUT = Path("/kaggle/working/results_part5")
OUT.mkdir(parents=True, exist_ok=True)

# ── 1. CONSTANTS ──────────────────────────────────────────────────────────────

HURST_BUDGETS     = [100, 200, 400, 800, 1600, 3200, 6400]
N_RUNS_PER_BUDGET = 5      # independent completions per problem per budget
N_PROBS           = 8      # problems per level
SESSION_LIMIT_MIN = 510

MODELS = {
    "phi35":  ("microsoft/Phi-3.5-mini-instruct", True),   # (model_id, has_system)
    "qwen3b": ("Qwen/Qwen2.5-3B-Instruct",        True),
}
MODEL_LABELS = {"phi35": "Phi-3.5-mini (3.8B)", "qwen3b": "Qwen2.5-3B (3.0B)"}
MODEL_COLS   = {"phi35": "#1B7837",              "qwen3b": "#762A83"}
LEVEL_COLS   = {"Level_1": "#2166AC", "Level_3": "#F4A582", "Level_5": "#D6604D"}
LEVEL_LABELS = {"Level_1": "Level 1 (easy)", "Level_3": "Level 3 (medium)",
                "Level_5": "Level 5 (hard)"}

SYS_PROMPT = (
    "You are a precise math assistant. "
    "Think step by step. "
    "Write your final numerical answer after the word ANSWER: on its own line."
)

MATH_PROBLEMS = {
    "Level_1": [
        ("Compute $2^3 + 3^2$.", "17"),
        ("What is $15\\%$ of $200$?", "30"),
        ("If $x + 5 = 12$, find $x$.", "7"),
        ("What is $\\sqrt{49}$?", "7"),
        ("Calculate $3!$.", "6"),
        ("What is $4^2 - 3^2$?", "7"),
        ("Find $x$: $2x = 18$.", "9"),
        ("Compute $7 \\times 8$.", "56"),
    ],
    "Level_3": [
        ("Solve $x^2 - 5x + 6 = 0$.", "2 or 3"),
        ("A circle has radius 7. Area in terms of pi.", "49*pi"),
        ("If $f(x)=2x^2-3x+1$, find $f(3)$.", "10"),
        ("Solve: $\\log_2 8 = x$.", "3"),
        ("Distance between $(0,0)$ and $(3,4)$.", "5"),
        ("What is $C(6,2)$?", "15"),
        ("Find the 10th term of $3,7,11,\\dots$", "39"),
        ("Solve $2^x=32$.", "5"),
    ],
    "Level_5": [
        ("Compute sum_{k=1}^{inf} 1/(k(k+1)).", "1"),
        ("Evaluate integral_0^pi sin^2(x) dx.", "pi/2"),
        ("Limit as x->0 of sin(3x)/x.", "3"),
        ("Eigenvalues of [[2,1],[1,2]].", "1 and 3"),
        ("Sum_{n=0}^{inf} 1/2^n.", "2"),
        ("Number of divisors of 360.", "24"),
        ("Solve e^{2x} - 3*e^x + 2 = 0.", "0 or log(2)"),
        ("d/dx of ln(x^2+1).", "2*x/(x**2+1)"),
    ],
}

# ── 2. CHECKPOINT I/O ─────────────────────────────────────────────────────────

CKPT_PATH = OUT / "checkpoint.json"

def load_checkpoint():
    if CKPT_PATH.exists():
        with open(CKPT_PATH) as f:
            return json.load(f)
    return {}

def save_checkpoint(ckpt):
    tmp = CKPT_PATH.with_suffix(".tmp")
    with open(tmp, "w") as f:
        json.dump(ckpt, f, indent=2, default=str)
    tmp.replace(CKPT_PATH)

def elapsed_min():
    return (time.time() - START_TIME) / 60

def time_ok():
    return elapsed_min() < SESSION_LIMIT_MIN

# ── 3. ANSWER CHECKING ────────────────────────────────────────────────────────

_SP_TRANSFORMS = standard_transformations + (implicit_multiplication_application,)

def _clean_latex(s):
    s = s.lower()
    s = re.sub(r"\\frac\s*{([^}]+)}\s*{([^}]+)}", r"(\1)/(\2)", s)
    s = re.sub(r"\\sqrt\s*{([^}]+)}", r"sqrt(\1)", s)
    for old, new in [
        (r"\\pi","pi"),(r"\\ln","log"),(r"\\log","log"),
        (r"\\sin","sin"),(r"\\cos","cos"),(r"\\cdot","*"),
        (r"\\times","*"),(r"\\infty","oo"),
    ]:
        s = re.sub(old, new, s)
    s = re.sub(r"[{}$\\]", " ", s)
    return s.strip()

def _sympy_eq(a, b):
    try:
        ea = parse_expr(a, transformations=_SP_TRANSFORMS)
        eb = parse_expr(b, transformations=_SP_TRANSFORMS)
        d  = sp.simplify(ea - eb)
        return d == 0 or abs(float(d.evalf())) < 1e-6
    except Exception:
        return False

def _extract_answer(text):
    for marker in ["ANSWER:", "Answer:", "= ", "equals "]:
        idx = text.rfind(marker)
        if idx >= 0:
            return text[idx + len(marker):].strip().split("\n")[0]
    return text.split(".")[-1].strip()

def answer_is_correct(generated, expected):
    gen_region = _extract_answer(generated)
    gen_clean  = _clean_latex(gen_region).replace(" ", "")
    parts = [p.strip() for p in expected.split("or") if p.strip()]
    for part in parts:
        part_clean = _clean_latex(part).replace(" ", "")
        if _sympy_eq(gen_clean, part_clean):
            return True
        if part_clean and part_clean in gen_clean:
            return True
    try:
        exp_num = float(sp.sympify(expected).evalf())
        nums = re.findall(r"-?\d+\.?\d*", gen_region)
        for n in nums:
            if abs(float(n) - exp_num) < 1e-4 * (abs(exp_num) + 1):
                return True
    except Exception:
        pass
    return False

# ── 4. MODEL HELPERS ──────────────────────────────────────────────────────────

def load_model_8bit(model_id):
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    bnb = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb,
        device_map="auto", trust_remote_code=False,
    )
    model.eval()
    return model, tok

def release(model):
    del model; gc.collect(); torch.cuda.empty_cache()

def build_prompt(tok, problem, sys_prompt, has_system):
    if has_system:
        messages = [{"role": "system", "content": sys_prompt},
                    {"role": "user",   "content": problem}]
    else:
        messages = [{"role": "user", "content": f"{sys_prompt}\n\n{problem}"}]
    try:
        text = tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        text = f"Instruct: {sys_prompt}\n{problem}\nOutput:"
    return tok(text, return_tensors="pt").to(DEVICE)

# ── 5. SINGLE-PROBLEM VARIANCE MEASUREMENT ───────────────────────────────────

def measure_problem_variance(model, tok, problem, budget, n_runs, has_system):
    """
    Run `n_runs` independent completions of `problem` at `budget` tokens.
    Returns:
      ent_var   : between-run variance of mean-token-entropy  (primary P5 signal)
      acc_var   : between-run variance of correctness (0/1)
      mean_ent  : mean of per-run mean-entropy
    """
    _, expected = problem
    prompt_str  = problem[0]
    inputs = build_prompt(tok, prompt_str, SYS_PROMPT, has_system)
    in_len = inputs["input_ids"].shape[1]

    run_ents    = []
    run_correct = []

    for _ in range(n_runs):
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=budget,
                do_sample=True,
                temperature=0.8,
                top_p=0.95,
                output_scores=True,
                return_dict_in_generate=True,
                pad_token_id=tok.eos_token_id,
            )
        gen_ids  = out.sequences[0][in_len:]
        gen_text = tok.decode(gen_ids, skip_special_tokens=True)
        run_correct.append(int(answer_is_correct(gen_text, expected)))

        if out.scores:
            ents = []
            for sc in out.scores:
                p = torch.softmax(sc[0], dim=-1).cpu().numpy()
                p = p[p > 1e-12]
                ents.append(float(-np.sum(p * np.log(p))))
            run_ents.append(float(np.mean(ents)))

    ent_var  = float(np.var(run_ents, ddof=1))    if len(run_ents) >= 2    else float("nan")
    acc_var  = float(np.var(run_correct, ddof=1)) if len(run_correct) >= 2 else float("nan")
    mean_ent = float(np.mean(run_ents))            if run_ents              else float("nan")
    return ent_var, acc_var, mean_ent

# ── 6. HURST ESTIMATION HELPERS ──────────────────────────────────────────────

def fit_hurst_ols(T_arr, V_arr):
    """
    Fit H from log V = a + 2H log T via OLS.
    Returns dict with H, slope, R2, p, se, CI_95.
    """
    valid = (~np.isnan(V_arr)) & (V_arr > 0) & (T_arr > 0)
    if valid.sum() < 4:
        return {"error": "insufficient_points", "n": int(valid.sum())}
    lT = np.log(T_arr[valid])
    lV = np.log(V_arr[valid])
    slope, intercept, r_val, p_val, se = stats.linregress(lT, lV)
    H = slope / 2.0
    # 95 % CI on slope → CI on H
    n_v   = int(valid.sum())
    t_crit = stats.t.ppf(0.975, df=n_v - 2)
    h_ci  = [float((slope - t_crit * se) / 2),
             float((slope + t_crit * se) / 2)]
    return {
        "H": float(H), "slope": float(slope),
        "intercept": float(intercept),
        "R2": float(r_val**2), "p": float(p_val),
        "se_slope": float(se), "H_CI_95": h_ci,
        "n": n_v,
        "supports_P5": bool(0.35 <= H <= 0.65),
    }

def bootstrap_hurst(T_arr, V_arr, n_boot=500):
    """Bootstrap CI on H (resampling problems, not time points)."""
    valid = (~np.isnan(V_arr)) & (V_arr > 0) & (T_arr > 0)
    lT = np.log(T_arr[valid])
    lV = np.log(V_arr[valid])
    if len(lT) < 4:
        return [float("nan"), float("nan")]
    rng = np.random.default_rng(99)
    slopes = []
    for _ in range(n_boot):
        idx = rng.choice(len(lT), len(lT), replace=True)
        if len(np.unique(idx)) < 3:
            continue
        try:
            sl, *_ = stats.linregress(lT[idx], lV[idx])
            slopes.append(sl)
        except Exception:
            continue
    if len(slopes) < 50:
        return [float("nan"), float("nan")]
    H_boot = np.array(slopes) / 2.0
    return [float(np.percentile(H_boot, 2.5)),
            float(np.percentile(H_boot, 97.5))]

# ── 7. MAIN EXPERIMENT LOOP ───────────────────────────────────────────────────

ckpt = load_checkpoint()
print(f"\n[Checkpoint] {len(ckpt)} entries already done.")

for model_key, (model_id, has_sys) in MODELS.items():
    if not time_ok():
        print(f"\n[TIME] approaching session limit — stopping.")
        break

    levels_needed = []
    for lv in MATH_PROBLEMS:
        budgets_needed = [b for b in HURST_BUDGETS
                          if f"{model_key}_{lv}_{b}" not in ckpt]
        if budgets_needed:
            levels_needed.append((lv, budgets_needed))

    if not levels_needed:
        print(f"\n[SKIP] {model_key} — all (level, budget) pairs done.")
        continue

    print(f"\n{'='*66}")
    print(f"  Loading {MODEL_LABELS[model_key]}  [{elapsed_min():.1f} min]")
    print(f"{'='*66}")

    try:
        model, tok = load_model_8bit(model_id)
        print(f"  Model loaded  [{elapsed_min():.1f} min]")
    except Exception as e:
        print(f"  ERROR: {e}")
        ckpt[f"{model_key}_load_error"] = str(e)
        save_checkpoint(ckpt)
        continue

    for level, budgets_needed in levels_needed:
        problems = MATH_PROBLEMS[level][:N_PROBS]

        for budget in budgets_needed:
            ck_key = f"{model_key}_{level}_{budget}"
            if ck_key in ckpt:
                continue
            if not time_ok():
                print(f"  [TIME] stopping before {ck_key}")
                break

            print(f"\n  [{model_key}][{level}]  T={budget}  "
                  f"{len(problems)} problems × {N_RUNS_PER_BUDGET} runs")

            prob_ent_vars = []
            prob_acc_vars = []
            prob_ents     = []

            for prob in problems:
                ev, av, me = measure_problem_variance(
                    model, tok, prob, budget, N_RUNS_PER_BUDGET, has_sys)
                if not math.isnan(ev):
                    prob_ent_vars.append(ev)
                if not math.isnan(av):
                    prob_acc_vars.append(av)
                if not math.isnan(me):
                    prob_ents.append(me)

            # Aggregate across problems: mean variance
            V_ent = float(np.mean(prob_ent_vars)) if prob_ent_vars else float("nan")
            V_acc = float(np.mean(prob_acc_vars)) if prob_acc_vars else float("nan")
            H_ent = float(np.mean(prob_ents))     if prob_ents     else float("nan")

            ckpt[ck_key] = {
                "budget": budget, "V_ent": V_ent, "V_acc": V_acc,
                "mean_ent": H_ent, "n_problems": len(prob_ent_vars),
                "elapsed_min": elapsed_min(),
            }
            save_checkpoint(ckpt)
            print(f"    V_ent={V_ent:.6f}  V_acc={V_acc:.4f}  "
                  f"[{elapsed_min():.1f} min]")

    release(model)
    print(f"  [released] {model_key}  [{elapsed_min():.1f} min]")

# ── 8. ASSEMBLE & FIT ─────────────────────────────────────────────────────────

RESULTS = {}

for model_key in MODELS:
    RESULTS[model_key] = {}
    for level in MATH_PROBLEMS:
        series = []
        for budget in HURST_BUDGETS:
            ck_key = f"{model_key}_{level}_{budget}"
            if ck_key in ckpt and "V_ent" in ckpt[ck_key]:
                series.append(ckpt[ck_key])
        if not series:
            continue

        T_arr = np.array([s["budget"] for s in series], dtype=float)
        V_arr = np.array([s["V_ent"]  for s in series], dtype=float)

        hurst_ols  = fit_hurst_ols(T_arr, V_arr)
        hurst_boot = bootstrap_hurst(T_arr, V_arr, n_boot=600)

        if "H" in hurst_ols:
            hurst_ols["H_CI_boot"] = hurst_boot

        RESULTS[model_key][level] = {
            "series": series,
            "T": T_arr.tolist(),
            "V_ent": V_arr.tolist(),
            "hurst": hurst_ols,
        }
        h = hurst_ols.get("H", float("nan"))
        r2 = hurst_ols.get("R2", float("nan"))
        ci = hurst_ols.get("H_CI_boot", ["?","?"])
        print(f"  [{model_key}][{level}]  H={h:.3f}  "
              f"CI=[{ci[0]:.3f},{ci[1]:.3f}]  R²={r2:.3f}  "
              f"P5={'✓' if hurst_ols.get('supports_P5') else '✗'}")

out_json = OUT / "hurst_results.json"
with open(out_json, "w") as f:
    json.dump(RESULTS, f, indent=2, default=str)
print(f"\n[json] hurst_results.json  [{elapsed_min():.1f} min]")

# ── 9. FIGURES ────────────────────────────────────────────────────────────────

plt.rcParams.update({
    "font.family":"serif","font.size":11,"axes.titlesize":12,
    "axes.labelsize":11,"legend.fontsize":9,"savefig.dpi":300,
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.alpha":0.25,
})

def savefig(name, fig=None):
    fig = fig or plt.gcf()
    for ext in ("pdf","png"):
        fig.savefig(OUT / f"{name}.{ext}", bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"  [fig] {name}")

# ── Figure C1: log V vs log T (one row per model, one col per level) ─────────
models_done = [mk for mk in MODELS if any(RESULTS.get(mk, {}).values())]
if models_done:
    n_rows = len(models_done)
    n_cols = len(MATH_PROBLEMS)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows),
                             squeeze=False)
    for ri, mk in enumerate(models_done):
        for ci_ax, level in enumerate(["Level_1","Level_3","Level_5"]):
            ax  = axes[ri][ci_ax]
            res = RESULTS.get(mk, {}).get(level, {})
            if not res:
                ax.set_visible(False); continue
            T_arr = np.array(res["T"], dtype=float)
            V_arr = np.array(res["V_ent"], dtype=float)
            valid = (~np.isnan(V_arr)) & (V_arr > 0)
            col   = MODEL_COLS[mk]
            if valid.sum() >= 2:
                ax.scatter(np.log10(T_arr[valid]), np.log10(V_arr[valid]),
                           color=col, s=60, zorder=5)
                hr = res.get("hurst", {})
                if "H" in hr:
                    lT = np.log10(T_arr[valid])
                    # OLS line in log10 space: log10(V) = intercept/ln10 + slope/ln10 * log10(T)
                    slope10 = hr["slope"]                     # this is d(lnV)/d(lnT)
                    inter10 = hr["intercept"]                 # intercept in ln space
                    lT_r    = np.array([lT.min(), lT.max()])
                    lV_r    = (inter10 + slope10 * np.log(10**lT_r)) / np.log(10)
                    h_val   = hr["H"]
                    ci_b    = hr.get("H_CI_boot", [float("nan"), float("nan")])
                    ax.plot(lT_r, lV_r, color=col, lw=2.0,
                            label=f"H={h_val:.3f} [{ci_b[0]:.2f},{ci_b[1]:.2f}]")
            # H=0.5 reference (slope=1 on log-log)
            lT_ref = np.array([np.log10(T_arr.min()), np.log10(T_arr.max())])
            ax.plot(lT_ref, lT_ref - lT_ref.mean() + np.log10(V_arr[valid].mean()),
                    "k:", lw=1.5, alpha=0.5, label="$H=0.5$ ref")
            ax.set_xlabel(r"$\log_{10}(T)$")
            ax.set_ylabel(r"$\log_{10}(V_{\mathrm{ent}})$")
            ax.set_title(f"{MODEL_LABELS[mk]}\n{LEVEL_LABELS[level]}")
            ax.legend(fontsize=8)
    fig.suptitle(
        r"Experiment C: $V(T)\propto T^{2H}$ — P5 Hurst exponent test"
        "\n(dots=data, line=OLS fit, dashed=H=0.5 reference)",
        fontsize=11, y=1.02,
    )
    plt.tight_layout()
    savefig("figure_hurst_loglog", fig)

# ── Figure C2: H estimates with CI bars (grouped bar) ────────────────────────
h_data = {}
for mk in models_done:
    h_data[mk] = {}
    for level in MATH_PROBLEMS:
        hr = RESULTS.get(mk, {}).get(level, {}).get("hurst", {})
        if "H" in hr:
            ci_b = hr.get("H_CI_boot", [float("nan"), float("nan")])
            h_data[mk][level] = {"H": hr["H"], "ci": ci_b}

if h_data:
    levels = ["Level_1","Level_3","Level_5"]
    x  = np.arange(len(levels))
    w  = 0.35 / max(len(models_done), 1)
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, mk in enumerate(models_done):
        Hs   = [h_data[mk].get(lv, {}).get("H",    float("nan")) for lv in levels]
        lo   = [h_data[mk].get(lv, {}).get("ci", [float("nan"),float("nan")])[0] for lv in levels]
        hi   = [h_data[mk].get(lv, {}).get("ci", [float("nan"),float("nan")])[1] for lv in levels]
        errl = [H - l if not math.isnan(H) and not math.isnan(l) else 0
                for H, l in zip(Hs, lo)]
        errh = [h - H if not math.isnan(H) and not math.isnan(h) else 0
                for H, h in zip(Hs, hi)]
        xs   = x + i * w - (len(models_done)-1) * w / 2
        ax.bar(xs, [h if not math.isnan(h) else 0 for h in Hs],
               w*0.9, color=MODEL_COLS[mk], alpha=0.8,
               label=MODEL_LABELS[mk], zorder=3)
        valid_mask = [not math.isnan(H) for H in Hs]
        if any(valid_mask):
            ax.errorbar(
                xs, [h if not math.isnan(h) else 0 for h in Hs],
                yerr=[errl, errh], fmt="none",
                ecolor="black", elinewidth=1.5, capsize=4, zorder=4,
            )
    ax.axhline(0.5, color="red", lw=1.5, ls="--", alpha=0.7, label="$H=0.5$ (theory)")
    ax.axhspan(0.35, 0.65, alpha=0.06, color="red", label="acceptance band [0.35, 0.65]")
    ax.set_xticks(x)
    ax.set_xticklabels([LEVEL_LABELS[lv] for lv in levels])
    ax.set_ylabel("Hurst exponent $H$")
    ax.set_ylim(0, 1.0)
    ax.set_title("Experiment C: H estimates with 95% bootstrap CI\n"
                 "P5 prediction: H ≈ 0.5 (random-walk variance growth)")
    ax.legend(fontsize=9)
    plt.tight_layout()
    savefig("figure_hurst_summary", fig)

# ── 10. LaTeX TABLE ───────────────────────────────────────────────────────────

def tex(v, fmt=".3f"):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return "—"
    return format(v, fmt)

lines = [
    "% AUTO-GENERATED by part5_hurst_extended.py",
    r"\begin{table}[t]",
    r"\centering\small",
    r"\caption{Experiment C (P5): Hurst exponent estimates. "
    r"$H$ fitted by OLS on $\log V_{\mathrm{ent}}$ vs $\log T$; "
    r"95\,\% CI from 600 bootstrap resamples. "
    r"P5 predicts $H\approx 0.5$; acceptance band $[0.35,0.65]$ shaded.}",
    r"\label{tab:hurst}",
    r"\begin{tabular}{llcccc}",
    r"\toprule",
    r"Model & Level & $\hat H$ & 95\% CI (boot) & $R^2$ & P5 \\",
    r"\midrule",
]
for mk in models_done:
    for li, level in enumerate(["Level_1","Level_3","Level_5"]):
        hr  = RESULTS.get(mk, {}).get(level, {}).get("hurst", {})
        mlbl = MODEL_LABELS[mk] if li == 0 else ""
        if "H" not in hr:
            lines.append(rf"{mlbl} & {level.replace('_',' ')} & — & — & — & — \\")
            continue
        ci_b = hr.get("H_CI_boot", [float("nan"), float("nan")])
        p5   = r"\checkmark" if hr.get("supports_P5") else r"\texttimes"
        lines.append(
            rf"{mlbl} & {level.replace('_',' ')} & "
            rf"${tex(hr['H'])}$ & "
            rf"$[{tex(ci_b[0])},{tex(ci_b[1])}]$ & "
            rf"${tex(hr.get('R2'))}$ & {p5} \\"
        )
    lines.append(r"\midrule")
lines[-1] = r"\bottomrule"
lines += [r"\end{tabular}", r"\end{table}"]
(OUT / "table_hurst.tex").write_text("\n".join(lines), encoding="utf-8")
print("  [tex] table_hurst.tex")

# ── 11. SUMMARY ───────────────────────────────────────────────────────────────

print("\n" + "="*66)
print("  EXPERIMENT C (P5 — HURST) SUMMARY")
print("="*66)
for mk in MODELS:
    for level in MATH_PROBLEMS:
        hr = RESULTS.get(mk, {}).get(level, {}).get("hurst", {})
        if "H" in hr:
            ci_b = hr.get("H_CI_boot", ["?","?"])
            print(f"  [{MODEL_LABELS[mk]}][{level}]  "
                  f"H={hr['H']:.3f}  CI=[{ci_b[0]:.3f},{ci_b[1]:.3f}]  "
                  f"R²={hr.get('R2',0):.3f}  "
                  f"P5={'SUPPORTED' if hr.get('supports_P5') else 'NOT SUPPORTED'}")
print(f"\n  Total elapsed: {elapsed_min():.1f} min")
print(f"  Outputs: {OUT}")
for fp in sorted(OUT.iterdir()):
    print(f"    {fp.name:48s}  {fp.stat().st_size/1024:7.1f} KB")

---
## Experiment D — Precision Sweep (P3)

**Prediction P3 tested:** T*(fp16) > T*(bf16) > T*(int8) > T*(int4).  
Phi-3.5-mini reloaded at each precision level; all three difficulty levels tested.


In [ ]:
import os, sys, json, time, math, warnings, gc, re
from pathlib import Path

warnings.filterwarnings("ignore")
START_TIME = time.time()

import subprocess
for pkg in ["torch", "transformers", "accelerate", "bitsandbytes", "scipy", "sympy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                   capture_output=True)

import numpy as np
import scipy.stats as stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import torch
import sympy as sp
from sympy.parsing.sympy_parser import (
    parse_expr, standard_transformations, implicit_multiplication_application,
)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

torch.manual_seed(42); np.random.seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Setup] device={DEVICE}  GPUs={torch.cuda.device_count()}")
if DEVICE == "cuda":
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB")

OUT = Path("/kaggle/working/results_part6")
OUT.mkdir(parents=True, exist_ok=True)

# ── 1. CONSTANTS ──────────────────────────────────────────────────────────────

MODEL_ID  = "microsoft/Phi-3.5-mini-instruct"
BUDGETS   = [300, 800, 1600, 3200, 6400]
N_SAMPLES = 3
N_PROBS   = 10
SESSION_LIMIT_MIN = 510

# Ordered from highest to lowest precision (P3 expects T* in same order)
PRECISIONS = ["fp16", "bf16", "int8", "int4"]
PREC_LABELS = {
    "fp16": "FP16 (full)",
    "bf16": "BF16 (brain float)",
    "int8": "INT8 (8-bit)",
    "int4": "INT4 (4-bit)",
}
PREC_COLS = {
    "fp16": "#2166AC",
    "bf16": "#4DAC26",
    "int8": "#F4A582",
    "int4": "#D6604D",
}
LEVEL_LABELS = {
    "Level_1": "Level 1 (easy)",
    "Level_3": "Level 3 (medium)",
    "Level_5": "Level 5 (hard)",
}
LEVEL_COLS = {"Level_1":"#2166AC","Level_3":"#F4A582","Level_5":"#D6604D"}

SYS_PROMPT = (
    "You are a precise math assistant. "
    "Think step by step. "
    "Write your final numerical answer after the word ANSWER: on its own line."
)

MATH_PROBLEMS = {
    "Level_1": [
        ("Compute $2^3 + 3^2$.", "17"),
        ("What is $15\\%$ of $200$?", "30"),
        ("If $x + 5 = 12$, find $x$.", "7"),
        ("What is $\\sqrt{49}$?", "7"),
        ("Calculate $3!$.", "6"),
        ("What is $4^2 - 3^2$?", "7"),
        ("Find $x$: $2x = 18$.", "9"),
        ("Compute $7 \\times 8$.", "56"),
        ("What is $2^5$?", "32"),
        ("Find the mean of $4, 6, 8, 10$.", "7"),
    ],
    "Level_3": [
        ("Solve $x^2 - 5x + 6 = 0$.", "2 or 3"),
        ("A circle has radius 7. Area in terms of pi.", "49*pi"),
        ("If $f(x)=2x^2-3x+1$, find $f(3)$.", "10"),
        ("Solve: $\\log_2 8 = x$.", "3"),
        ("Distance between $(0,0)$ and $(3,4)$.", "5"),
        ("What is $C(6,2)$?", "15"),
        ("Find the 10th term of $3,7,11,\\dots$", "39"),
        ("Solve $|x-3|=5$.", "8 or -2"),
        ("Solve $2^x=32$.", "5"),
        ("Sum of first 10 natural numbers.", "55"),
    ],
    "Level_5": [
        ("Compute sum_{k=1}^{inf} 1/(k(k+1)).", "1"),
        ("Evaluate integral_0^pi sin^2(x) dx.", "pi/2"),
        ("Limit as x->0 of sin(3x)/x.", "3"),
        ("Eigenvalues of [[2,1],[1,2]].", "1 and 3"),
        ("Sum_{n=0}^{inf} 1/2^n.", "2"),
        ("Number of divisors of 360.", "24"),
        ("Solve e^{2x} - 3*e^x + 2 = 0.", "0 or log(2)"),
        ("d/dx of ln(x^2+1).", "2*x/(x**2+1)"),
        ("Sum 1^2 + 2^2 + ... + 10^2.", "385"),
        ("Solve x^4 - 5*x^2 + 4 = 0.", "1 or 2"),
    ],
}

# ── 2. CHECKPOINT I/O ─────────────────────────────────────────────────────────

CKPT_PATH = OUT / "checkpoint.json"

def load_checkpoint():
    if CKPT_PATH.exists():
        with open(CKPT_PATH) as f:
            return json.load(f)
    return {}

def save_checkpoint(ckpt):
    tmp = CKPT_PATH.with_suffix(".tmp")
    with open(tmp, "w") as f:
        json.dump(ckpt, f, indent=2, default=str)
    tmp.replace(CKPT_PATH)

def elapsed_min():
    return (time.time() - START_TIME) / 60

def time_ok():
    return elapsed_min() < SESSION_LIMIT_MIN

# ── 3. ANSWER CHECKING ────────────────────────────────────────────────────────

_SP_TRANSFORMS = standard_transformations + (implicit_multiplication_application,)

def _clean_latex(s):
    s = s.lower()
    s = re.sub(r"\\frac\s*{([^}]+)}\s*{([^}]+)}", r"(\1)/(\2)", s)
    s = re.sub(r"\\sqrt\s*{([^}]+)}", r"sqrt(\1)", s)
    for old, new in [
        (r"\\pi","pi"),(r"\\ln","log"),(r"\\log","log"),
        (r"\\sin","sin"),(r"\\cos","cos"),(r"\\cdot","*"),
        (r"\\times","*"),(r"\\infty","oo"),
    ]:
        s = re.sub(old, new, s)
    s = re.sub(r"[{}$\\]", " ", s)
    return s.strip()

def _sympy_eq(a, b):
    try:
        ea = parse_expr(a, transformations=_SP_TRANSFORMS)
        eb = parse_expr(b, transformations=_SP_TRANSFORMS)
        d  = sp.simplify(ea - eb)
        return d == 0 or abs(float(d.evalf())) < 1e-6
    except Exception:
        return False

def _extract_answer(text):
    for marker in ["ANSWER:", "Answer:", "= ", "equals "]:
        idx = text.rfind(marker)
        if idx >= 0:
            return text[idx + len(marker):].strip().split("\n")[0]
    return text.split(".")[-1].strip()

def answer_is_correct(generated, expected):
    gen_region = _extract_answer(generated)
    gen_clean  = _clean_latex(gen_region).replace(" ", "")
    parts = [p.strip() for p in expected.split("or") if p.strip()]
    for part in parts:
        part_clean = _clean_latex(part).replace(" ", "")
        if _sympy_eq(gen_clean, part_clean):
            return True
        if part_clean and part_clean in gen_clean:
            return True
    try:
        exp_num = float(sp.sympify(expected).evalf())
        nums = re.findall(r"-?\d+\.?\d*", gen_region)
        for n in nums:
            if abs(float(n) - exp_num) < 1e-4 * (abs(exp_num) + 1):
                return True
    except Exception:
        pass
    return False

# ── 4. g-MODEL FIT ────────────────────────────────────────────────────────────

def g_raw(T, Delta, sigma_r2, k):
    V = np.maximum(k * np.asarray(T, dtype=float), 0.0)
    s = np.maximum(sigma_r2 + V, 1e-12)
    return (1.0 / np.sqrt(2.0 * np.pi * s)) * np.exp(-Delta**2 / (2.0 * s))

def g_scaled(T, Delta, sigma_r2, k, scale, offset):
    return scale * g_raw(T, Delta, sigma_r2, k) + offset

def fit_g(T_arr, A_arr, n_boot=400):
    T = np.asarray(T_arr, dtype=float)
    A = np.asarray(A_arr, dtype=float)
    best_r2, best_p = -np.inf, None
    for Di in [0.3, 0.5, 0.8, 1.2, 2.0]:
        for ki in [1e-5, 5e-5, 1e-4, 5e-4]:
            try:
                T_pk = T[np.argmax(A)]
                gp   = g_raw(np.array([T_pk]), Di, 0.05, ki)[0]
                sc0  = max(A) / max(gp, 1e-10)
                popt, _ = curve_fit(
                    g_scaled, T, A,
                    p0=[Di, 0.05, ki, sc0, 0.0],
                    bounds=([0.01,1e-4,1e-9, 1.0,-50.0],
                            [5.00,5.00,1.00,1e6, 50.0]),
                    maxfev=20000, method="trf",
                )
                ss_r = np.sum((A - g_scaled(T, *popt))**2)
                ss_t = np.sum((A - A.mean())**2)
                r2   = 1.0 - ss_r / ss_t if ss_t > 0 else -np.inf
                if r2 > best_r2:
                    best_r2, best_p = r2, popt
            except Exception:
                continue
    if best_p is None:
        return {"error": "fit_failed"}
    Delta, sr2, k, sc, off = best_p
    T_fine = np.linspace(T.min(), T.max() * 2.5, 8000)
    A_fine = g_scaled(T_fine, *best_p)
    T_star = float(T_fine[np.argmax(A_fine)])
    peak   = float(np.max(A_fine))
    n   = len(T)
    rss = float(np.sum((A - g_scaled(T, *best_p))**2))
    kp  = len(best_p)
    aic = n * np.log(rss / n + 1e-15) + 2 * kp
    bic = n * np.log(rss / n + 1e-15) + kp * np.log(n)
    # bootstrap CI on T*
    ci  = {}
    bts = []
    rng = np.random.default_rng(42)
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        Tb, Ab = T[idx], A[idx]
        if len(np.unique(Tb)) < 3:
            continue
        try:
            pb, _ = curve_fit(
                g_scaled, Tb, Ab, p0=best_p,
                bounds=([0.01,1e-4,1e-9, 1.0,-50.0],
                        [5.00,5.00,1.00,1e6, 50.0]),
                maxfev=5000, method="trf",
            )
            bts.append(pb)
        except Exception:
            continue
    if len(bts) >= 40:
        ba = np.array(bts)
        bt_s = []
        for pb in ba:
            Tf2 = np.linspace(T.min(), T.max() * 2.5, 2000)
            bt_s.append(float(Tf2[np.argmax(g_scaled(Tf2, *pb))]))
        ci["T_star"] = [float(np.percentile(bt_s, 2.5)),
                        float(np.percentile(bt_s, 97.5))]
    return {
        "Delta":float(Delta),"sigma_r2":float(sr2),"k":float(k),
        "scale":float(sc),"offset":float(off),
        "R2":float(best_r2),"T_star":T_star,"peak_acc":peak,
        "AIC":float(aic),"BIC":float(bic),
        "CI":ci,"n_data":int(n),
    }

# ── 5. MODEL LOADING ──────────────────────────────────────────────────────────

def load_model_at_precision(model_id, prec):
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    if prec == "fp16":
        model = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=torch.float16,
            device_map="auto", trust_remote_code=False,
        )
    elif prec == "bf16":
        model = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=torch.bfloat16,
            device_map="auto", trust_remote_code=False,
        )
    elif prec == "int8":
        bnb = BitsAndBytesConfig(load_in_8bit=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb,
            device_map="auto", trust_remote_code=False,
        )
    elif prec == "int4":
        bnb = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb,
            device_map="auto", trust_remote_code=False,
        )
    else:
        raise ValueError(f"Unknown precision: {prec}")

    model.eval()
    return model, tok

def release(model):
    del model; gc.collect(); torch.cuda.empty_cache()

def build_prompt(tok, problem):
    messages = [
        {"role": "system", "content": SYS_PROMPT},
        {"role": "user",   "content": problem},
    ]
    try:
        text = tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        text = f"<|system|>\n{SYS_PROMPT}<|end|>\n<|user|>\n{problem}<|end|>\n<|assistant|>\n"
    return tok(text, return_tensors="pt").to(DEVICE)

# ── 6. SWEEP ──────────────────────────────────────────────────────────────────

def run_sweep(model, tok, problems, budgets, n_samples=N_SAMPLES):
    results = []
    for budget in budgets:
        all_correct = []
        for problem, expected in problems:
            inputs = build_prompt(tok, problem)
            in_len = inputs["input_ids"].shape[1]
            run_correct = []
            for _ in range(n_samples):
                with torch.no_grad():
                    out = model.generate(
                        **inputs,
                        max_new_tokens=budget,
                        do_sample=True, temperature=0.7, top_p=0.9,
                        pad_token_id=tok.eos_token_id,
                        return_dict_in_generate=True,
                    )
                # out is GenerateOutput if return_dict_in_generate=True, else raw tensor
                seq = out.sequences if hasattr(out, "sequences") else out
                gen_ids  = seq[0][in_len:]
                gen_text = tok.decode(gen_ids, skip_special_tokens=True)
                run_correct.append(int(answer_is_correct(gen_text, expected)))
            all_correct.append(float(np.mean(run_correct)))
        acc = float(np.mean(all_correct))
        results.append({"budget": budget, "accuracy": acc})
        print(f"      T={budget:5d}  acc={acc:.3f}")
    return results

# ── 7. MAIN EXPERIMENT LOOP ───────────────────────────────────────────────────

ckpt = load_checkpoint()
print(f"\n[Checkpoint] {len(ckpt)} entries already done.")

for prec in PRECISIONS:
    if not time_ok():
        print(f"\n[TIME] approaching session limit — stopping.")
        break

    levels_needed = [lv for lv in MATH_PROBLEMS
                     if f"{prec}_{lv}" not in ckpt]
    if not levels_needed:
        print(f"\n[SKIP] {prec} — all levels done.")
        continue

    print(f"\n{'='*66}")
    print(f"  Loading {MODEL_ID} at {prec.upper()}  [{elapsed_min():.1f} min]")
    print(f"  Levels remaining: {levels_needed}")
    print(f"{'='*66}")

    try:
        model, tok = load_model_at_precision(MODEL_ID, prec)
        print(f"  Loaded  [{elapsed_min():.1f} min]")
    except Exception as e:
        print(f"  ERROR loading {prec}: {e}")
        ckpt[f"{prec}_load_error"] = str(e)
        save_checkpoint(ckpt)
        continue

    for level in levels_needed:
        ck_key = f"{prec}_{level}"
        if ck_key in ckpt:
            continue
        if not time_ok():
            print(f"  [TIME] stopping before {ck_key}")
            break

        problems = MATH_PROBLEMS[level][:N_PROBS]
        print(f"\n  [{prec}][{level}]  "
              f"{len(problems)} probs × {len(BUDGETS)} budgets × {N_SAMPLES} runs")

        sweep = run_sweep(model, tok, problems, BUDGETS)
        T_arr = np.array([r["budget"]       for r in sweep])
        A_arr = np.array([r["accuracy"]*100 for r in sweep])
        fit   = fit_g(T_arr, A_arr)

        print(f"  -> R²={fit.get('R2',0):.4f}  T*={fit.get('T_star',0):.0f}")

        ckpt[ck_key] = {"sweep": sweep, "fit": fit,
                        "prec": prec, "level": level,
                        "elapsed_min": elapsed_min()}
        save_checkpoint(ckpt)
        print(f"  [ckpt] {ck_key}  [{elapsed_min():.1f} min]")

    release(model)
    print(f"  [released] {prec}  [{elapsed_min():.1f} min]")

# ── 8. ASSEMBLE RESULTS ───────────────────────────────────────────────────────

RESULTS = {"sweeps": {}, "fits": {}, "P3_tests": {}}

for prec in PRECISIONS:
    RESULTS["sweeps"][prec] = {}
    RESULTS["fits"][prec]   = {}
    for level in MATH_PROBLEMS:
        ck_key = f"{prec}_{level}"
        if ck_key in ckpt and "sweep" in ckpt[ck_key]:
            RESULTS["sweeps"][prec][level] = ckpt[ck_key]["sweep"]
            RESULTS["fits"][prec][level]   = ckpt[ck_key]["fit"]

# P3 test per level: T* monotone-decreasing with quantisation?
for level in MATH_PROBLEMS:
    t_stars = {}
    for prec in PRECISIONS:
        fit = RESULTS["fits"].get(prec, {}).get(level, {})
        if "T_star" in fit:
            t_stars[prec] = fit["T_star"]
    if len(t_stars) >= 2:
        avail = [p for p in PRECISIONS if p in t_stars]
        mono  = all(t_stars[avail[i]] >= t_stars[avail[i+1]]
                    for i in range(len(avail)-1))
        RESULTS["P3_tests"][level] = {
            "T_stars": t_stars,
            "precisions_available": avail,
            "monotone_decreasing": mono,
        }
        print(f"  [{level}] P3: "
              + "  ".join(f"{p}={t_stars[p]:.0f}" for p in avail)
              + f"  supported={'✓' if mono else '✗'}")

out_json = OUT / "precision_results.json"
with open(out_json, "w") as f:
    json.dump(RESULTS, f, indent=2, default=str)
print(f"\n[json] precision_results.json  [{elapsed_min():.1f} min]")

# ── 9. FIGURES ────────────────────────────────────────────────────────────────

plt.rcParams.update({
    "font.family":"serif","font.size":11,"axes.titlesize":12,
    "axes.labelsize":11,"legend.fontsize":9,"savefig.dpi":300,
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.alpha":0.25,
})

def savefig(name, fig=None):
    fig = fig or plt.gcf()
    for ext in ("pdf","png"):
        fig.savefig(OUT / f"{name}.{ext}", bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"  [fig] {name}")

# ── Figure D1: Level_3 curves for all 4 precisions ───────────────────────────
level_main = "Level_3"
has_any = any(level_main in RESULTS["sweeps"].get(p, {}) for p in PRECISIONS)
if has_any:
    fig, ax = plt.subplots(figsize=(8, 4.8))
    T_dense = np.linspace(BUDGETS[0], BUDGETS[-1]*1.4, 2000)
    for prec in PRECISIONS:
        data = RESULTS["sweeps"].get(prec, {}).get(level_main)
        if not data:
            continue
        col = PREC_COLS[prec]
        T_d = np.array([r["budget"]       for r in data])
        A_d = np.array([r["accuracy"]*100 for r in data])
        ax.scatter(T_d/1000, A_d, color=col, s=50, zorder=6)
        ax.plot(T_d/1000, A_d, "o-", color=col, lw=1.3, alpha=0.4)
        fit = RESULTS["fits"].get(prec, {}).get(level_main, {})
        if "Delta" in fit:
            popt = [fit["Delta"],fit["sigma_r2"],fit["k"],fit["scale"],fit["offset"]]
            Tf   = T_dense[T_dense <= T_d.max()*1.3]
            ax.plot(Tf/1000, g_scaled(Tf, *popt), color=col, lw=2.2,
                    label=f"{PREC_LABELS[prec]}  T*={fit['T_star']:.0f}")
            ax.axvline(fit["T_star"]/1000, color=col, lw=0.9, ls=":", alpha=0.7)
    ax.set_xlabel("Budget $T$ (thousands of tokens)")
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("Experiment D: Precision sweep — Level 3 (medium)\n"
                 r"P3 prediction: $T^*(\mathrm{fp16})>T^*(\mathrm{bf16})>"
                 r"T^*(\mathrm{int8})>T^*(\mathrm{int4})$")
    ax.legend(fontsize=9)
    plt.tight_layout()
    savefig("figure_precision_curves", fig)

# ── Figure D2: T* bar chart across all levels ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax_i, level in enumerate(["Level_1","Level_3","Level_5"]):
    ax  = axes[ax_i]
    p3t = RESULTS["P3_tests"].get(level, {})
    ts  = p3t.get("T_stars", {})
    avail = [p for p in PRECISIONS if p in ts]
    if not avail:
        ax.set_visible(False); continue
    vals = [ts[p] for p in avail]
    cols = [PREC_COLS[p] for p in avail]
    lbls = [PREC_LABELS[p] for p in avail]
    bars = ax.bar(range(len(avail)), vals, color=cols, alpha=0.82,
                  width=0.6, zorder=3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+max(vals)*0.02,
                f"{v:.0f}", ha="center", fontsize=9, fontweight="bold")
    ax.set_xticks(range(len(avail)))
    ax.set_xticklabels(lbls, fontsize=8, rotation=15)
    ax.set_ylabel("$T^*$ (tokens)")
    mono = p3t.get("monotone_decreasing", False)
    ax.set_title(f"{LEVEL_LABELS[level]}\nP3 {'✓ supported' if mono else '✗ not supported'}")
    ax.grid(axis="x", alpha=0)
plt.suptitle("Experiment D: T* vs precision (Phi-3.5-mini)\n"
             "P3 predicts monotone decrease from FP16 → INT4",
             fontsize=11, y=1.02)
plt.tight_layout()
savefig("figure_precision_tstar", fig)

# ── 10. LaTeX TABLE ───────────────────────────────────────────────────────────

def tex(v, fmt=".3f"):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return "—"
    return format(v, fmt)

lines = [
    "% AUTO-GENERATED by part6_precision_extended.py",
    r"\begin{table}[t]",
    r"\centering\small",
    r"\caption{Experiment D (P3): overthinking threshold $T^*$ by weight "
    r"precision and difficulty level (Phi-3.5-mini, 10 problems/level, "
    r"3 runs/budget). P3 predicts $T^*$ monotone-decreasing from FP16 to INT4. "
    r"\checkmark = supported.}",
    r"\label{tab:precision-extended}",
    r"\begin{tabular}{lcccc}",
    r"\toprule",
    r"Level & FP16 & BF16 & INT8 & INT4 \\",
    r"\midrule",
]
for level in ["Level_1","Level_3","Level_5"]:
    vals = []
    for prec in PRECISIONS:
        fit = RESULTS["fits"].get(prec, {}).get(level, {})
        vals.append(f"${tex(fit.get('T_star'),'.0f')}$" if "T_star" in fit else "—")
    p3t  = RESULTS["P3_tests"].get(level, {})
    mono = r"\checkmark" if p3t.get("monotone_decreasing") else r"\texttimes"
    lines.append(
        rf"{LEVEL_LABELS[level]} & {' & '.join(vals)} \\ "
        rf"\quad[P3: {mono}]"
    )
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
(OUT / "table_precision_extended.tex").write_text("\n".join(lines), encoding="utf-8")
print("  [tex] table_precision_extended.tex")

# ── 11. SUMMARY ───────────────────────────────────────────────────────────────

print("\n" + "="*66)
print("  EXPERIMENT D (P3 — PRECISION SWEEP) SUMMARY")
print("="*66)
for level in ["Level_1","Level_3","Level_5"]:
    p3t = RESULTS["P3_tests"].get(level, {})
    ts  = p3t.get("T_stars", {})
    mono = p3t.get("monotone_decreasing", "?")
    row  = "  ".join(f"{p}={ts[p]:.0f}" for p in PRECISIONS if p in ts)
    print(f"  {level}: {row}  P3={'SUPPORTED' if mono else 'NOT SUPPORTED'}")
print(f"\n  Total elapsed: {elapsed_min():.1f} min")
print(f"  Outputs: {OUT}")
for fp in sorted(OUT.iterdir()):
    print(f"    {fp.name:48s}  {fp.stat().st_size/1024:7.1f} KB")

---
## Experiment E — Prospective Out-of-Sample Prediction

**Test:** Fit g-model on token budgets [150,300,600,1200]; predict accuracy at unseen budgets [2400,4800,9600].  
Strong result: R²_test > 0.8 demonstrates the model captures a real phenomenon, not curve-fitting artefact.


In [ ]:
import os, sys, json, time, math, warnings, gc, re
from pathlib import Path

warnings.filterwarnings("ignore")
START_TIME = time.time()

import subprocess
for pkg in ["torch", "transformers", "accelerate", "bitsandbytes", "scipy", "sympy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                   capture_output=True)

import numpy as np
import scipy.stats as stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import torch
import sympy as sp
from sympy.parsing.sympy_parser import (
    parse_expr, standard_transformations, implicit_multiplication_application,
)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

torch.manual_seed(42); np.random.seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Setup] device={DEVICE}  GPUs={torch.cuda.device_count()}")
if DEVICE == "cuda":
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB")

OUT = Path("/kaggle/working/results_part7")
OUT.mkdir(parents=True, exist_ok=True)

# ── 1. CONSTANTS ──────────────────────────────────────────────────────────────

TRAIN_BUDGETS  = [150, 300, 600, 1200]
TEST_BUDGETS   = [2400, 4800, 9600]
ALL_BUDGETS    = TRAIN_BUDGETS + TEST_BUDGETS

N_SAMPLES = 3
N_PROBS   = 10
SESSION_LIMIT_MIN = 510

MODELS = {
    "phi35":  ("microsoft/Phi-3.5-mini-instruct", True),
    "qwen3b": ("Qwen/Qwen2.5-3B-Instruct",        True),
}
MODEL_LABELS = {"phi35": "Phi-3.5-mini (3.8B)", "qwen3b": "Qwen2.5-3B (3.0B)"}
MODEL_COLS   = {"phi35": "#1B7837",              "qwen3b": "#762A83"}
LEVEL_COLS   = {"Level_1":"#2166AC","Level_3":"#F4A582","Level_5":"#D6604D"}
LEVEL_LABELS = {"Level_1":"Level 1 (easy)","Level_3":"Level 3 (medium)",
                "Level_5":"Level 5 (hard)"}

SYS_PROMPT = (
    "You are a precise math assistant. "
    "Think step by step. "
    "Write your final numerical answer after the word ANSWER: on its own line."
)

MATH_PROBLEMS = {
    "Level_1": [
        ("Compute $2^3 + 3^2$.", "17"),
        ("What is $15\\%$ of $200$?", "30"),
        ("If $x + 5 = 12$, find $x$.", "7"),
        ("What is $\\sqrt{49}$?", "7"),
        ("Calculate $3!$.", "6"),
        ("What is $4^2 - 3^2$?", "7"),
        ("Find $x$: $2x = 18$.", "9"),
        ("Compute $7 \\times 8$.", "56"),
        ("What is $2^5$?", "32"),
        ("Find the mean of $4, 6, 8, 10$.", "7"),
    ],
    "Level_3": [
        ("Solve $x^2 - 5x + 6 = 0$.", "2 or 3"),
        ("A circle has radius 7. Area in terms of pi.", "49*pi"),
        ("If $f(x)=2x^2-3x+1$, find $f(3)$.", "10"),
        ("Solve: $\\log_2 8 = x$.", "3"),
        ("Distance between $(0,0)$ and $(3,4)$.", "5"),
        ("What is $C(6,2)$?", "15"),
        ("Find the 10th term of $3,7,11,\\dots$", "39"),
        ("Solve $|x-3|=5$.", "8 or -2"),
        ("Solve $2^x=32$.", "5"),
        ("Sum of first 10 natural numbers.", "55"),
    ],
    "Level_5": [
        ("Compute sum_{k=1}^{inf} 1/(k(k+1)).", "1"),
        ("Evaluate integral_0^pi sin^2(x) dx.", "pi/2"),
        ("Limit as x->0 of sin(3x)/x.", "3"),
        ("Eigenvalues of [[2,1],[1,2]].", "1 and 3"),
        ("Sum_{n=0}^{inf} 1/2^n.", "2"),
        ("Number of divisors of 360.", "24"),
        ("Solve e^{2x} - 3*e^x + 2 = 0.", "0 or log(2)"),
        ("d/dx of ln(x^2+1).", "2*x/(x**2+1)"),
        ("Sum 1^2 + 2^2 + ... + 10^2.", "385"),
        ("Solve x^4 - 5*x^2 + 4 = 0.", "1 or 2"),
    ],
}

# ── 2. CHECKPOINT I/O ─────────────────────────────────────────────────────────

CKPT_PATH = OUT / "checkpoint.json"

def load_checkpoint():
    if CKPT_PATH.exists():
        with open(CKPT_PATH) as f:
            return json.load(f)
    return {}

def save_checkpoint(ckpt):
    tmp = CKPT_PATH.with_suffix(".tmp")
    with open(tmp, "w") as f:
        json.dump(ckpt, f, indent=2, default=str)
    tmp.replace(CKPT_PATH)

def elapsed_min():
    return (time.time() - START_TIME) / 60

def time_ok():
    return elapsed_min() < SESSION_LIMIT_MIN

# ── 3. ANSWER CHECKING ────────────────────────────────────────────────────────

_SP_TRANSFORMS = standard_transformations + (implicit_multiplication_application,)

def _clean_latex(s):
    s = s.lower()
    s = re.sub(r"\\frac\s*{([^}]+)}\s*{([^}]+)}", r"(\1)/(\2)", s)
    s = re.sub(r"\\sqrt\s*{([^}]+)}", r"sqrt(\1)", s)
    for old, new in [
        (r"\\pi","pi"),(r"\\ln","log"),(r"\\log","log"),
        (r"\\sin","sin"),(r"\\cos","cos"),(r"\\cdot","*"),
        (r"\\times","*"),(r"\\infty","oo"),
    ]:
        s = re.sub(old, new, s)
    s = re.sub(r"[{}$\\]", " ", s)
    return s.strip()

def _sympy_eq(a, b):
    try:
        ea = parse_expr(a, transformations=_SP_TRANSFORMS)
        eb = parse_expr(b, transformations=_SP_TRANSFORMS)
        d  = sp.simplify(ea - eb)
        return d == 0 or abs(float(d.evalf())) < 1e-6
    except Exception:
        return False

def _extract_answer(text):
    for marker in ["ANSWER:", "Answer:", "= ", "equals "]:
        idx = text.rfind(marker)
        if idx >= 0:
            return text[idx + len(marker):].strip().split("\n")[0]
    return text.split(".")[-1].strip()

def answer_is_correct(generated, expected):
    gen_region = _extract_answer(generated)
    gen_clean  = _clean_latex(gen_region).replace(" ", "")
    parts = [p.strip() for p in expected.split("or") if p.strip()]
    for part in parts:
        part_clean = _clean_latex(part).replace(" ", "")
        if _sympy_eq(gen_clean, part_clean):
            return True
        if part_clean and part_clean in gen_clean:
            return True
    try:
        exp_num = float(sp.sympify(expected).evalf())
        nums = re.findall(r"-?\d+\.?\d*", gen_region)
        for n in nums:
            if abs(float(n) - exp_num) < 1e-4 * (abs(exp_num) + 1):
                return True
    except Exception:
        pass
    return False

# ── 4. g-MODEL FIT (train-only) ───────────────────────────────────────────────

def g_raw(T, Delta, sigma_r2, k):
    V = np.maximum(k * np.asarray(T, dtype=float), 0.0)
    s = np.maximum(sigma_r2 + V, 1e-12)
    return (1.0 / np.sqrt(2.0 * np.pi * s)) * np.exp(-Delta**2 / (2.0 * s))

def g_scaled(T, Delta, sigma_r2, k, scale, offset):
    return scale * g_raw(T, Delta, sigma_r2, k) + offset

def fit_g_train(T_train, A_train, n_boot=300):
    """
    Fit g-model on training budgets only.
    Returns params, T*, and prediction function.
    """
    T = np.asarray(T_train, dtype=float)
    A = np.asarray(A_train, dtype=float)
    best_r2, best_p = -np.inf, None

    for Di in [0.3, 0.5, 0.8, 1.2, 2.0]:
        for ki in [1e-5, 5e-5, 1e-4, 5e-4]:
            try:
                T_pk = T[np.argmax(A)]
                gp   = g_raw(np.array([T_pk]), Di, 0.05, ki)[0]
                sc0  = max(A) / max(gp, 1e-10)
                popt, _ = curve_fit(
                    g_scaled, T, A,
                    p0=[Di, 0.05, ki, sc0, 0.0],
                    bounds=([0.01,1e-4,1e-9, 1.0,-50.0],
                            [5.00,5.00,1.00,1e6, 50.0]),
                    maxfev=20000, method="trf",
                )
                ss_r = np.sum((A - g_scaled(T, *popt))**2)
                ss_t = np.sum((A - A.mean())**2)
                r2   = 1.0 - ss_r / ss_t if ss_t > 0 else -np.inf
                if r2 > best_r2:
                    best_r2, best_p = r2, popt
            except Exception:
                continue

    if best_p is None:
        return None

    T_fine = np.linspace(T.min(), max(T.max(), max(TEST_BUDGETS)) * 1.1, 10000)
    A_fine = g_scaled(T_fine, *best_p)
    T_star = float(T_fine[np.argmax(A_fine)])

    # bootstrap CI on params using training data only
    rng  = np.random.default_rng(42)
    bts  = []
    n    = len(T)
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        Tb, Ab = T[idx], A[idx]
        if len(np.unique(Tb)) < 3:
            continue
        try:
            pb, _ = curve_fit(
                g_scaled, Tb, Ab, p0=best_p,
                bounds=([0.01,1e-4,1e-9, 1.0,-50.0],
                        [5.00,5.00,1.00,1e6, 50.0]),
                maxfev=5000, method="trf",
            )
            bts.append(pb)
        except Exception:
            continue

    # Predict at TEST budgets using each bootstrap sample → prediction CI
    T_test = np.array(TEST_BUDGETS, dtype=float)
    pred_boot = []
    for pb in bts:
        pred_boot.append(g_scaled(T_test, *pb).tolist())

    pred_ci = {}
    if pred_boot:
        pb_arr = np.array(pred_boot)
        for i, b in enumerate(TEST_BUDGETS):
            pred_ci[b] = [float(np.percentile(pb_arr[:, i], 2.5)),
                          float(np.percentile(pb_arr[:, i], 97.5))]

    return {
        "params":     best_p.tolist(),
        "R2_train":   float(best_r2),
        "T_star":     T_star,
        "pred_ci":    pred_ci,
        "n_boot_ok":  len(bts),
    }

def predict_at(params, budgets):
    """Apply fitted params to arbitrary budgets."""
    return g_scaled(np.array(budgets, dtype=float), *params).tolist()

def eval_prediction(pred, actual):
    """MAE, RMSE, R² for test-set prediction."""
    p = np.array(pred)
    a = np.array(actual)
    mae  = float(np.mean(np.abs(p - a)))
    rmse = float(np.sqrt(np.mean((p - a)**2)))
    ss_r = float(np.sum((p - a)**2))
    ss_t = float(np.sum((a - a.mean())**2))
    r2   = float(1.0 - ss_r / ss_t) if ss_t > 0 else float("nan")
    return {"MAE": mae, "RMSE": rmse, "R2_test": r2}

# ── 5. MODEL HELPERS ──────────────────────────────────────────────────────────

def load_model_8bit(model_id):
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    bnb = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb,
        device_map="auto", trust_remote_code=False,
    )
    model.eval()
    return model, tok

def release(model):
    del model; gc.collect(); torch.cuda.empty_cache()

def build_prompt(tok, problem, has_system):
    if has_system:
        messages = [{"role":"system","content":SYS_PROMPT},
                    {"role":"user",  "content":problem}]
    else:
        messages = [{"role":"user","content":f"{SYS_PROMPT}\n\n{problem}"}]
    try:
        text = tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        text = f"Instruct: {SYS_PROMPT}\n{problem}\nOutput:"
    return tok(text, return_tensors="pt").to(DEVICE)

def run_one_budget(model, tok, problems, budget, has_sys, n_samples=N_SAMPLES):
    all_correct = []
    for problem, expected in problems:
        inputs = build_prompt(tok, problem, has_sys)
        in_len = inputs["input_ids"].shape[1]
        run_correct = []
        for _ in range(n_samples):
            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=budget,
                    do_sample=True, temperature=0.7, top_p=0.9,
                    pad_token_id=tok.eos_token_id,
                    return_dict_in_generate=True,
                )
            seq = out.sequences if hasattr(out, "sequences") else out
            gen_ids  = seq[0][in_len:]
            gen_text = tok.decode(gen_ids, skip_special_tokens=True)
            run_correct.append(int(answer_is_correct(gen_text, expected)))
        all_correct.append(float(np.mean(run_correct)))
    return float(np.mean(all_correct))

# ── 6. MAIN EXPERIMENT LOOP ───────────────────────────────────────────────────

ckpt = load_checkpoint()
print(f"\n[Checkpoint] {len(ckpt)} (model,level,budget) entries already done.")

for model_key, (model_id, has_sys) in MODELS.items():
    if not time_ok():
        print(f"\n[TIME] approaching session limit — stopping.")
        break

    # Find which (level, budget) pairs are still needed
    todo = []
    for lv in MATH_PROBLEMS:
        for b in ALL_BUDGETS:
            if f"{model_key}_{lv}_{b}" not in ckpt:
                todo.append((lv, b))

    if not todo:
        print(f"\n[SKIP] {model_key} — all done.")
        continue

    print(f"\n{'='*66}")
    print(f"  Loading {MODEL_LABELS[model_key]}  [{elapsed_min():.1f} min]")
    print(f"  Remaining: {len(todo)} (level,budget) pairs")
    print(f"{'='*66}")

    try:
        model, tok = load_model_8bit(model_id)
        print(f"  Model loaded  [{elapsed_min():.1f} min]")
    except Exception as e:
        print(f"  ERROR: {e}")
        ckpt[f"{model_key}_load_error"] = str(e)
        save_checkpoint(ckpt)
        continue

    for level, budget in todo:
        ck_key = f"{model_key}_{level}_{budget}"
        if ck_key in ckpt:
            continue
        if not time_ok():
            print(f"  [TIME] stopping before {ck_key}")
            break

        tag = "TRAIN" if budget in TRAIN_BUDGETS else "TEST "
        print(f"  [{model_key}][{level}][{tag} T={budget}]  "
              f"{N_PROBS} probs × {N_SAMPLES} runs")

        problems = MATH_PROBLEMS[level][:N_PROBS]
        acc = run_one_budget(model, tok, problems, budget, has_sys)

        ckpt[ck_key] = {
            "accuracy": acc, "budget": budget,
            "split": "train" if budget in TRAIN_BUDGETS else "test",
            "elapsed_min": elapsed_min(),
        }
        save_checkpoint(ckpt)
        print(f"    acc={acc:.3f}  [{elapsed_min():.1f} min]")

    release(model)
    print(f"  [released] {model_key}  [{elapsed_min():.1f} min]")

# ── 7. ASSEMBLE & PREDICT ─────────────────────────────────────────────────────

RESULTS = {}

for model_key in MODELS:
    RESULTS[model_key] = {}
    for level in MATH_PROBLEMS:
        # Collect measured accuracies
        measured = {}
        for b in ALL_BUDGETS:
            ck_key = f"{model_key}_{level}_{b}"
            if ck_key in ckpt and "accuracy" in ckpt[ck_key]:
                measured[b] = ckpt[ck_key]["accuracy"] * 100  # as %

        if len(measured) < len(TRAIN_BUDGETS):
            print(f"  [{model_key}][{level}] insufficient train data — skipping fit")
            RESULTS[model_key][level] = {"measured": measured, "error": "insufficient_train"}
            continue

        # Fit on train budgets only
        T_train = np.array([b for b in TRAIN_BUDGETS if b in measured])
        A_train = np.array([measured[b] for b in TRAIN_BUDGETS if b in measured])

        fit_res = fit_g_train(T_train, A_train)
        if fit_res is None:
            print(f"  [{model_key}][{level}] fit failed")
            RESULTS[model_key][level] = {"measured": measured, "error": "fit_failed"}
            continue

        # Predict at test budgets using train-fitted params
        T_test_available = [b for b in TEST_BUDGETS if b in measured]
        predicted_test   = predict_at(fit_res["params"], T_test_available)
        actual_test      = [measured[b] for b in T_test_available]

        eval_res = eval_prediction(predicted_test, actual_test) \
                   if len(actual_test) >= 2 else {}

        RESULTS[model_key][level] = {
            "measured":     {str(k): v for k, v in measured.items()},
            "fit":          fit_res,
            "T_test_avail": T_test_available,
            "predicted":    {b: p for b, p in zip(T_test_available, predicted_test)},
            "actual_test":  {b: a for b, a in zip(T_test_available, actual_test)},
            "eval":         eval_res,
        }

        r2t = eval_res.get("R2_test", float("nan"))
        mae = eval_res.get("MAE", float("nan"))
        print(f"  [{MODEL_LABELS[model_key]}][{level}]  "
              f"R²_train={fit_res['R2_train']:.3f}  "
              f"R²_test={r2t:.3f}  MAE={mae:.1f}%  "
              f"T*={fit_res['T_star']:.0f}")

out_json = OUT / "prospective_results.json"
with open(out_json, "w") as f:
    json.dump(RESULTS, f, indent=2, default=str)
print(f"\n[json] prospective_results.json  [{elapsed_min():.1f} min]")

# ── 8. FIGURES ────────────────────────────────────────────────────────────────

plt.rcParams.update({
    "font.family":"serif","font.size":11,"axes.titlesize":11,
    "axes.labelsize":11,"legend.fontsize":9,"savefig.dpi":300,
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.alpha":0.25,
})

def savefig(name, fig=None):
    fig = fig or plt.gcf()
    for ext in ("pdf","png"):
        fig.savefig(OUT / f"{name}.{ext}", bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"  [fig] {name}")

models_done = [mk for mk in MODELS
               if any(RESULTS.get(mk,{}).get(lv,{}).get("fit") for lv in MATH_PROBLEMS)]

# ── Figure E1: train fit + test prediction (one row per model, one col per level) ──
if models_done:
    n_rows = len(models_done)
    n_cols = 3
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5*n_cols, 4.8*n_rows),
                             squeeze=False)
    for ri, mk in enumerate(models_done):
        for ci_ax, level in enumerate(["Level_1","Level_3","Level_5"]):
            ax  = axes[ri][ci_ax]
            res = RESULTS.get(mk, {}).get(level, {})
            if not res or "fit" not in res or not res["fit"]:
                ax.set_visible(False); continue

            measured  = {int(k): v for k, v in res["measured"].items()}
            fit_res   = res["fit"]
            predicted = {int(k): v for k, v in res.get("predicted", {}).items()}
            actual_t  = {int(k): v for k, v in res.get("actual_test", {}).items()}

            col = MODEL_COLS[mk]
            # All measured data
            T_all = np.array(sorted(measured.keys()), dtype=float)
            A_all = np.array([measured[int(b)] for b in T_all])

            # Train points
            T_tr = np.array([b for b in T_all if int(b) in TRAIN_BUDGETS])
            A_tr = np.array([measured[int(b)] for b in T_tr])

            # Test actual points
            T_te = np.array(sorted(actual_t.keys()), dtype=float)
            A_te = np.array([actual_t[int(b)] for b in T_te])

            # Fitted curve (extrapolated to test range)
            T_dense = np.linspace(min(T_all)*0.8, max(T_all)*1.05, 3000)
            A_curve = g_scaled(T_dense, *fit_res["params"])

            ax.scatter(T_tr/1000, A_tr, color=col, s=60, zorder=7,
                       marker="o", label="Train data")
            ax.scatter(T_te/1000, A_te, color="black", s=80, zorder=8,
                       marker="*", label="Test actual")
            ax.plot(T_dense/1000, A_curve, color=col, lw=2.0, zorder=5,
                    label=f"g-fit (train only)\nT*={fit_res['T_star']:.0f}")

            # Predicted test points
            for b, p_val in predicted.items():
                ax.scatter(b/1000, p_val, color="red", s=80, marker="^",
                           zorder=9)
                if int(b) in actual_t:
                    ax.plot([b/1000, b/1000], [p_val, actual_t[int(b)]],
                            "r-", lw=1.2, alpha=0.7)

            # Prediction CI bands
            ci = fit_res.get("pred_ci", {})
            for b_str, ci_vals in ci.items():
                b_int = int(b_str)
                if ci_vals[0] is not None:
                    ax.axvspan(b_int/1000 - 0.05, b_int/1000 + 0.05,
                               ymin=(ci_vals[0] - ax.get_ylim()[0]) /
                                    max(ax.get_ylim()[1] - ax.get_ylim()[0], 1),
                               ymax=(ci_vals[1] - ax.get_ylim()[0]) /
                                    max(ax.get_ylim()[1] - ax.get_ylim()[0], 1),
                               alpha=0.12, color="red")

            # Vertical split line
            ax.axvline(max(TRAIN_BUDGETS)/1000, color="gray", lw=1.2,
                       ls="--", alpha=0.6, label="train|test split")

            eval_r = res.get("eval", {})
            r2t    = eval_r.get("R2_test", float("nan"))
            mae    = eval_r.get("MAE",     float("nan"))
            ax.set_xlabel("Budget $T$ (k tokens)")
            ax.set_ylabel("Accuracy (%)")
            ax.set_title(
                f"{MODEL_LABELS[mk]}\n{LEVEL_LABELS[level]}\n"
                f"R²_test={r2t:.2f}  MAE={mae:.1f}%"
            )
            ax.legend(fontsize=7, loc="upper right")

    fig.suptitle(
        "Experiment E: Prospective prediction\n"
        "Model fitted on T∈{150,300,600,1200} — predicts T∈{2400,4800,9600}\n"
        "● train  ★ test-actual  ▲ test-predicted   red line = error",
        fontsize=11, y=1.03,
    )
    plt.tight_layout()
    savefig("figure_prospective_fit", fig)

# ── Figure E2: prediction error summary ───────────────────────────────────────
error_data = []
for mk in models_done:
    for level in MATH_PROBLEMS:
        res  = RESULTS.get(mk, {}).get(level, {})
        eval_r = res.get("eval", {})
        if "R2_test" in eval_r:
            error_data.append({
                "model": mk, "level": level,
                "R2_test": eval_r["R2_test"],
                "MAE":     eval_r["MAE"],
                "RMSE":    eval_r["RMSE"],
            })

if error_data:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
    levels    = ["Level_1","Level_3","Level_5"]
    x         = np.arange(len(levels))
    w         = 0.35
    for ax_idx, (metric, label, thresh) in enumerate([
        ("R2_test", "$R^2$ on test budgets",     0.8),
        ("MAE",     "MAE (pp) on test budgets",  None),
    ]):
        ax = axes[ax_idx]
        for i, mk in enumerate(models_done):
            vals = []
            for lv in levels:
                d = next((e for e in error_data if e["model"]==mk and e["level"]==lv), {})
                vals.append(d.get(metric, float("nan")))
            xs = x + (i - (len(models_done)-1)/2) * w
            ax.bar(xs, [v if not math.isnan(v) else 0 for v in vals],
                   w*0.9, color=MODEL_COLS[mk], alpha=0.82,
                   label=MODEL_LABELS[mk], zorder=3)
        if thresh is not None:
            ax.axhline(thresh, color="red", lw=1.5, ls="--",
                       alpha=0.7, label=f"threshold={thresh}")
        ax.set_xticks(x)
        ax.set_xticklabels([LEVEL_LABELS[lv] for lv in levels], fontsize=9)
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.legend(fontsize=8)
    plt.suptitle("Experiment E: Prospective prediction quality\n"
                 "R² > 0.8 → model captures real phenomenon (not just overfitting)",
                 fontsize=11, y=1.02)
    plt.tight_layout()
    savefig("figure_prospective_error", fig)

# ── 9. LaTeX TABLE ────────────────────────────────────────────────────────────

def tex(v, fmt=".3f"):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return "—"
    return format(v, fmt)

lines = [
    "% AUTO-GENERATED by part7_prospective.py",
    r"\begin{table}[t]",
    r"\centering\small",
    r"\caption{Experiment E: prospective prediction quality. "
    r"g-model fitted on $T\in\{150,300,600,1200\}$ tokens only, "
    r"then evaluated at unseen $T\in\{2400,4800,9600\}$. "
    r"$R^2_{\mathrm{test}}>0.8$ indicates the theory generalises beyond "
    r"the training window.}",
    r"\label{tab:prospective}",
    r"\begin{tabular}{llcccc}",
    r"\toprule",
    r"Model & Level & $R^2_{\mathrm{train}}$ & $R^2_{\mathrm{test}}$ "
    r"& MAE (pp) & $T^*$ \\",
    r"\midrule",
]
for mk in MODELS:
    for li, level in enumerate(["Level_1","Level_3","Level_5"]):
        res    = RESULTS.get(mk, {}).get(level, {})
        fit_r  = res.get("fit", {}) or {}
        eval_r = res.get("eval", {}) or {}
        mlbl   = MODEL_LABELS[mk] if li == 0 else ""
        if not fit_r:
            lines.append(rf"{mlbl} & {level.replace('_',' ')} & \multicolumn{{4}}{{c}}{{incomplete}} \\")
        else:
            lines.append(
                rf"{mlbl} & {level.replace('_',' ')} & "
                rf"${tex(fit_r.get('R2_train'))}$ & "
                rf"${tex(eval_r.get('R2_test'))}$ & "
                rf"${tex(eval_r.get('MAE'),'.1f')}$ & "
                rf"${tex(fit_r.get('T_star'),'.0f')}$ \\"
            )
    lines.append(r"\midrule")
lines[-1] = r"\bottomrule"
lines += [r"\end{tabular}", r"\end{table}"]
(OUT / "table_prospective.tex").write_text("\n".join(lines), encoding="utf-8")
print("  [tex] table_prospective.tex")

# ── 10. SUMMARY ───────────────────────────────────────────────────────────────

print("\n" + "="*66)
print("  EXPERIMENT E (PROSPECTIVE PREDICTION) SUMMARY")
print("="*66)
for mk in MODELS:
    for level in MATH_PROBLEMS:
        res    = RESULTS.get(mk, {}).get(level, {})
        fit_r  = res.get("fit", {}) or {}
        eval_r = res.get("eval", {}) or {}
        if fit_r:
            r2tr = fit_r.get("R2_train", float("nan"))
            r2te = eval_r.get("R2_test",  float("nan"))
            mae  = eval_r.get("MAE",      float("nan"))
            tstar= fit_r.get("T_star",    float("nan"))
            strong = "STRONG" if r2te > 0.8 else ("MODERATE" if r2te > 0.5 else "WEAK")
            print(f"  [{MODEL_LABELS[mk]}][{level}]  "
                  f"R²_tr={r2tr:.3f}  R²_te={r2te:.3f}  "
                  f"MAE={mae:.1f}pp  T*={tstar:.0f}  → {strong}")
print(f"\n  Total elapsed: {elapsed_min():.1f} min")
print(f"  Outputs: {OUT}")
for fp in sorted(OUT.iterdir()):
    print(f"    {fp.name:48s}  {fp.stat().st_size/1024:7.1f} KB")

---
## Final Merge — Summary Figures and Paper Tables

Reads all JSON outputs from Experiments A–E and produces:
- `figure_summary_predictions.png` — six-panel overview of all predictions
- `figure_tstar_heatmap.png` — T* heatmap across models × difficulty levels
- `table_predictions_summary.tex` / `table_prospective_summary.tex` — drop-in LaTeX tables


In [ ]:
import json, math, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import scipy.stats as stats

OUT = DIRS['merged']
OUT.mkdir(parents=True, exist_ok=True)

# ── COLOUR PALETTE ────────────────────────────────────────────────────────────

BLUE   = "#2166AC"
RED    = "#D6604D"
ORANGE = "#F4A582"
GREEN  = "#4DAC26"
PURPLE = "#762A83"
GRAY   = "#878787"
GOLD   = "#B35806"

LEVEL_COLS  = {"Level_1": BLUE,   "Level_3": ORANGE, "Level_5": RED}
LEVEL_LBLS  = {"Level_1": "L1 (easy)", "Level_3": "L3 (med)", "Level_5": "L5 (hard)"}
MODEL_COLS  = {
    "phi35":    GREEN,
    "qwen3b":   PURPLE,
    "gemma2":   GOLD,
    "qwen_1b5": "#1F78B4",
    "qwen_3b":  "#33A02C",
    "qwen_7b":  "#E31A1C",
}
MODEL_LBLS  = {
    "phi35":    "Phi-3.5-mini",
    "qwen3b":   "Qwen2.5-3B",
    "gemma2":   "Gemma-2-2B",
    "qwen_1b5": "Qwen1.5B",
    "qwen_3b":  "Qwen3B",
    "qwen_7b":  "Qwen7B",
}
PREC_COLS  = {"fp16": BLUE, "bf16": GREEN, "int8": ORANGE, "int4": RED}
PREC_LBLS  = {"fp16":"FP16","bf16":"BF16","int8":"INT8","int4":"INT4"}

plt.rcParams.update({
    "font.family":"serif","font.size":10,"axes.titlesize":11,
    "axes.labelsize":10,"legend.fontsize":8,"savefig.dpi":300,
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.alpha":0.22,
    "lines.linewidth":1.8,"lines.markersize":5,
})

def savefig(name, fig=None):
    fig = fig or plt.gcf()
    for ext in ("pdf","png"):
        fig.savefig(OUT / f"{name}.{ext}", bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"  [fig] {name}")

def load_json(path):
    p = Path(path)
    if p.exists():
        with open(p) as f:
            return json.load(f)
    print(f"  [missing] {path}")
    return {}

def tex(v, fmt=".3f"):
    if v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))):
        return "—"
    return format(v, fmt)

# ── LOAD ALL RESULTS ──────────────────────────────────────────────────────────

R2  = load_json("/kaggle/working/results/expA/part2_results.json")
R3  = load_json("/kaggle/working/results/expA/multimodel_results.json")
R4  = load_json("/kaggle/working/results/expB/p6_results.json")
R5  = load_json("/kaggle/working/results/expC/hurst_results.json")
R6  = load_json("/kaggle/working/results/expD/precision_results.json")
R7  = load_json("/kaggle/working/results/expE/prospective_results.json")

MERGED = {
    "part2_original": R2,
    "exp_A_multimodel": R3,
    "exp_B_p6_scaling": R4,
    "exp_C_hurst": R5,
    "exp_D_precision": R6,
    "exp_E_prospective": R7,
}
with open(OUT / "all_results_merged.json", "w") as f:
    json.dump(MERGED, f, indent=2, default=str)
print("[json] all_results_merged.json")

# ── HELPER: extract T* from any result dict ───────────────────────────────────

def get_tstar(fit_dict):
    if not fit_dict or "T_star" not in fit_dict:
        return float("nan")
    return float(fit_dict["T_star"])

def get_r2(fit_dict):
    if not fit_dict or "R2" not in fit_dict:
        return float("nan")
    return float(fit_dict["R2"])

# ── FIGURE 1: GRAND SUMMARY — 5-panel, one per prediction ────────────────────

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# ─── Panel (a): P1 — T* by difficulty, multi-model ───────────────────────────
ax = fig.add_subplot(gs[0, 0])
levels  = ["Level_1","Level_3","Level_5"]
models_A = ["phi35","qwen3b","gemma2"]
x = np.arange(len(levels))
w = 0.28

for i, mk in enumerate(models_A):
    fits = R3.get("fits", {}).get(mk, {})
    vals = [get_tstar(fits.get(lv)) for lv in levels]
    valid_vals = [v for v in vals if not math.isnan(v)]
    if not valid_vals:
        continue
    bars = ax.bar(x + (i-1)*w, [v if not math.isnan(v) else 0 for v in vals],
                  w*0.9, color=MODEL_COLS[mk], alpha=0.82,
                  label=MODEL_LBLS[mk], zorder=3)
ax.set_xticks(x)
ax.set_xticklabels([LEVEL_LBLS[lv] for lv in levels], fontsize=9)
ax.set_ylabel("Optimal budget $T^*$ (tokens)")
ax.set_title("(a) P1: $T^*(d)$ increases with difficulty\n[Exp A — 3 models]")
ax.legend(fontsize=7, loc="upper left")

# ─── Panel (b): P3 — T* by precision, Level_3 ────────────────────────────────
ax = fig.add_subplot(gs[0, 1])
precs = ["fp16","bf16","int8","int4"]
p3_tests = R6.get("P3_tests", {})
ts_by_level = {}
for lv in levels:
    p3t = p3_tests.get(lv, {})
    ts_by_level[lv] = {p: p3t.get("T_stars",{}).get(p, float("nan")) for p in precs}

x2  = np.arange(len(precs))
w2  = 0.28
for i, lv in enumerate(levels):
    vals = [ts_by_level[lv].get(p, float("nan")) for p in precs]
    ax.bar(x2 + (i-1)*w2, [v if not math.isnan(v) else 0 for v in vals],
           w2*0.9, color=LEVEL_COLS[lv], alpha=0.82,
           label=LEVEL_LBLS[lv], zorder=3)
ax.set_xticks(x2)
ax.set_xticklabels([PREC_LBLS[p] for p in precs], fontsize=9)
ax.set_ylabel("$T^*$ (tokens)")
ax.set_title("(b) P3: $T^*$ decreases with quantisation\n[Exp D — FP16→BF16→INT8→INT4]")
ax.legend(fontsize=7)

# ─── Panel (c): P4 — peak accuracy vs difficulty ─────────────────────────────
ax = fig.add_subplot(gs[0, 2])
for i, mk in enumerate(models_A):
    fits = R3.get("fits", {}).get(mk, {})
    peaks = [fits.get(lv, {}).get("peak_acc", float("nan")) for lv in levels]
    valid = [(j, p) for j, p in enumerate(peaks) if not math.isnan(p)]
    if not valid:
        continue
    jj, pp = zip(*valid)
    ax.plot([LEVEL_LBLS[levels[j]] for j in jj], list(pp),
            "o-", color=MODEL_COLS[mk], lw=1.8, ms=7, label=MODEL_LBLS[mk])
ax.set_ylabel("Peak accuracy at $T^*$ (%)")
ax.set_title("(c) P4: peak accuracy decreases with difficulty\n[Exp A — 3 models]")
ax.legend(fontsize=7)

# ─── Panel (d): P5 — H estimates with CI ─────────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
hurst_models = ["phi35","qwen3b"]
x3  = np.arange(len(levels))
w3  = 0.35
for i, mk in enumerate(hurst_models):
    Hs   = []
    errl = []
    errh = []
    for lv in levels:
        hr = R5.get(mk, {}).get(lv, {}).get("hurst", {})
        H  = hr.get("H", float("nan"))
        ci = hr.get("H_CI_boot", [float("nan"), float("nan")])
        Hs.append(H if not math.isnan(H) else 0)
        errl.append((H - ci[0]) if not math.isnan(H) and not math.isnan(ci[0]) else 0)
        errh.append((ci[1] - H) if not math.isnan(H) and not math.isnan(ci[1]) else 0)
    xs = x3 + (i - 0.5) * w3
    ax.bar(xs, Hs, w3*0.9, color=MODEL_COLS[mk], alpha=0.82,
           label=MODEL_LBLS[mk], zorder=3)
    ax.errorbar(xs, Hs, yerr=[errl, errh], fmt="none",
                ecolor="black", elinewidth=1.5, capsize=4, zorder=4)
ax.axhline(0.5, color="red", lw=1.5, ls="--", alpha=0.7, label="H=0.5 (theory)")
ax.axhspan(0.35, 0.65, alpha=0.06, color="red")
ax.set_xticks(x3)
ax.set_xticklabels([LEVEL_LBLS[lv] for lv in levels], fontsize=9)
ax.set_ylabel("Hurst exponent $H$")
ax.set_ylim(0, 1.0)
ax.set_title("(d) P5: $H\\approx 0.5$ (variance grows as $T^{2H}$)\n[Exp C — 2 models]")
ax.legend(fontsize=7)

# ─── Panel (e): P6 — T* vs model size ────────────────────────────────────────
ax = fig.add_subplot(gs[1, 1])
qwen_keys  = ["qwen_1b5","qwen_3b","qwen_7b"]
qwen_params = [1500, 3000, 7000]
for lv in levels:
    ts_vals = []
    for mk in qwen_keys:
        t = get_tstar(R4.get("fits",{}).get(mk,{}).get(lv,{}))
        ts_vals.append(t)
    valid = [(np.log10(qwen_params[i]), ts_vals[i])
             for i in range(len(ts_vals)) if not math.isnan(ts_vals[i])]
    if not valid:
        continue
    xs, ys = zip(*valid)
    ax.plot(list(xs), list(ys), "o-", color=LEVEL_COLS[lv], lw=1.8, ms=7,
            label=LEVEL_LBLS[lv])
    if len(xs) >= 2:
        m, b_int, _, _, _ = stats.linregress(list(xs), list(ys))
        xr = np.array([min(xs), max(xs)])
        ax.plot(xr, m*xr+b_int, "--", color=LEVEL_COLS[lv], lw=1.0, alpha=0.5)
ax.set_xlabel(r"$\log_{10}(N)$ (params in M)")
ax.set_ylabel("$T^*$ (tokens)")
ax.set_title("(e) P6: $T^*$ decreases with model size\n"
             "[Exp B — Qwen2.5 family]")
ax.legend(fontsize=7)

# ─── Panel (f): Exp E — prospective R² summary ───────────────────────────────
ax = fig.add_subplot(gs[1, 2])
prosp_models = ["phi35","qwen3b"]
x4  = np.arange(len(levels))
w4  = 0.35
for i, mk in enumerate(prosp_models):
    r2_vals = [
        R7.get(mk,{}).get(lv,{}).get("eval",{}).get("R2_test", float("nan"))
        for lv in levels
    ]
    ax.bar(x4 + (i - 0.5)*w4,
           [v if not math.isnan(v) else 0 for v in r2_vals],
           w4*0.9, color=MODEL_COLS[mk], alpha=0.82,
           label=MODEL_LBLS[mk], zorder=3)
ax.axhline(0.8, color="red", lw=1.5, ls="--", alpha=0.7, label="$R^2=0.8$ threshold")
ax.set_xticks(x4)
ax.set_xticklabels([LEVEL_LBLS[lv] for lv in levels], fontsize=9)
ax.set_ylabel("$R^2$ on test budgets {2400,4800,9600}")
ax.set_title("(f) Prospective prediction quality\n"
             "[Exp E — fit on {150,300,600,1200}]")
ax.legend(fontsize=7)

fig.suptitle(
    "Summary of all falsifiable predictions (P1–P6) across experiments A–E\n"
    "Phi-3.5-mini, Qwen2.5 family, Gemma-2-2B  |  2×T4, 8-bit quantisation",
    fontsize=12, y=1.01,
)
savefig("figure_summary_predictions", fig)

# ── FIGURE 2: T* HEATMAP ──────────────────────────────────────────────────────

# Collect T* for all (model, level) pairs from Exp A
all_model_keys = ["phi35","qwen3b","gemma2"]
tstar_matrix   = np.full((len(all_model_keys), len(levels)), float("nan"))
for ri, mk in enumerate(all_model_keys):
    for ci_ax, lv in enumerate(levels):
        t = get_tstar(R3.get("fits",{}).get(mk,{}).get(lv,{}))
        tstar_matrix[ri, ci_ax] = t

# Fill any missing from part2 (Phi-only)
if math.isnan(tstar_matrix[0, 0]):
    for ci_ax, lv in enumerate(levels):
        t = get_tstar(R2.get("overthinking_fits",{}).get(lv,{}))
        if not math.isnan(t):
            tstar_matrix[0, ci_ax] = t

if not np.all(np.isnan(tstar_matrix)):
    vmin = np.nanmin(tstar_matrix)
    vmax = np.nanmax(tstar_matrix)
    fig, ax = plt.subplots(figsize=(7, 4))
    im  = ax.imshow(tstar_matrix, aspect="auto", cmap="YlOrRd",
                    vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(levels)))
    ax.set_xticklabels([LEVEL_LBLS[lv] for lv in levels])
    ax.set_yticks(range(len(all_model_keys)))
    ax.set_yticklabels([MODEL_LBLS[mk] for mk in all_model_keys])
    for ri in range(len(all_model_keys)):
        for ci_ax in range(len(levels)):
            v = tstar_matrix[ri, ci_ax]
            if not math.isnan(v):
                ax.text(ci_ax, ri, f"{v:.0f}", ha="center", va="center",
                        fontsize=11, fontweight="bold",
                        color="white" if v > (vmin + (vmax-vmin)*0.6) else "black")
    plt.colorbar(im, ax=ax, label="$T^*$ (tokens)")
    ax.set_title("T* heatmap — model × difficulty level\n"
                 "P1 (horizontal increase) and P4 confirmed if row-wise decrease absent")
    plt.tight_layout()
    savefig("figure_tstar_heatmap", fig)

# ── LaTeX: MAIN PREDICTIONS SUMMARY TABLE ────────────────────────────────────

def make_predictions_table():
    lines = [
        "% AUTO-GENERATED by part8_merge_figures.py",
        r"\begin{table}[t]",
        r"\centering\small",
        r"\caption{Summary of all falsifiable predictions tested. "
        r"\checkmark = supported; $\sim$ = mixed; \texttimes = not supported. "
        r"Each row corresponds to one proposition from the theory.}",
        r"\label{tab:predictions-summary}",
        r"\begin{tabular}{clp{5.5cm}l}",
        r"\toprule",
        r"Prop. & Test & Evidence & Result \\",
        r"\midrule",
    ]

    # P1
    p1_results = []
    for mk in ["phi35","qwen3b","gemma2"]:
        p1 = R3.get("P1_per_model",{}).get(mk,{})
        if p1:
            p1_results.append(p1.get("monotonic", False))
    p1_p2 = R2.get("P1_test",{})
    if p1_p2:
        p1_results.append(p1_p2.get("monotonic", False))
    p1_ok = sum(p1_results) / max(len(p1_results), 1)
    p1_sym = r"\checkmark" if p1_ok >= 0.67 else (r"$\sim$" if p1_ok >= 0.33 else r"\texttimes")
    lines.append(
        rf"P1 & $T^*(d)$ monotone-increasing & "
        rf"Exp A: {sum(p1_results)}/{len(p1_results)} models supported & {p1_sym} \\"
    )

    # P3
    p3_ok_count = 0; p3_total = 0
    for lv in ["Level_1","Level_3","Level_5"]:
        p3t = R6.get("P3_tests",{}).get(lv,{})
        if p3t:
            p3_total += 1
            if p3t.get("monotone_decreasing"):
                p3_ok_count += 1
    p3_sym = r"\checkmark" if p3_ok_count == p3_total and p3_total > 0 \
             else (r"$\sim$" if p3_ok_count > 0 else r"\texttimes")
    lines.append(
        rf"P3 & $T^*$ decreases with quantisation & "
        rf"Exp D: {p3_ok_count}/{p3_total} levels supported (FP16>BF16>INT8>INT4) & {p3_sym} \\"
    )

    # P4
    p4_results = []
    for mk in ["phi35","qwen3b","gemma2"]:
        fits = R3.get("fits",{}).get(mk,{})
        peaks = {lv: fits.get(lv,{}).get("peak_acc", float("nan"))
                 for lv in ["Level_1","Level_3","Level_5"]}
        if all(not math.isnan(v) for v in peaks.values()):
            ok = peaks["Level_1"] > peaks["Level_3"] > peaks["Level_5"]
            p4_results.append(ok)
    p4_sym = r"\checkmark" if sum(p4_results) == len(p4_results) and p4_results \
             else (r"$\sim$" if sum(p4_results) > 0 else r"\texttimes")
    lines.append(
        rf"P4 & Peak accuracy $\downarrow$ with difficulty & "
        rf"Exp A: {sum(p4_results)}/{len(p4_results)} models supported & {p4_sym} \\"
    )

    # P5
    h_ok = 0; h_total = 0
    for mk in ["phi35","qwen3b"]:
        for lv in ["Level_1","Level_3","Level_5"]:
            hr = R5.get(mk,{}).get(lv,{}).get("hurst",{})
            if "H" in hr:
                h_total += 1
                if hr.get("supports_P5"):
                    h_ok += 1
    p5_sym = r"\checkmark" if h_ok >= h_total*0.67 and h_total > 0 \
             else (r"$\sim$" if h_ok > 0 else r"\texttimes")
    lines.append(
        rf"P5 & $V(T)\propto T^{{2H}}$, $H\approx 0.5$ & "
        rf"Exp C: $H\approx 0.5$ in {h_ok}/{h_total} (model,level) pairs & {p5_sym} \\"
    )

    # P6
    p6_ok = 0; p6_total = 0
    for lv in ["Level_1","Level_3","Level_5"]:
        p6t = R4.get("P6_tests",{}).get(lv,{})
        if p6t:
            p6_total += 1
            if p6t.get("monotone_decreasing_with_N"):
                p6_ok += 1
    p6_sym = r"\checkmark" if p6_ok == p6_total and p6_total > 0 \
             else (r"$\sim$" if p6_ok > 0 else r"\texttimes")
    lines.append(
        rf"P6 & $\partial T^*/\partial N < 0$ (larger model $\to$ lower $T^*$) & "
        rf"Exp B: {p6_ok}/{p6_total} levels supported (Qwen 1.5B/3B/7B) & {p6_sym} \\"
    )

    # Prospective
    prosp_strong = 0; prosp_total = 0
    for mk in ["phi35","qwen3b"]:
        for lv in ["Level_1","Level_3","Level_5"]:
            r2t = R7.get(mk,{}).get(lv,{}).get("eval",{}).get("R2_test", float("nan"))
            if not math.isnan(r2t):
                prosp_total += 1
                if r2t > 0.8:
                    prosp_strong += 1
    prosp_sym = r"\checkmark" if prosp_strong >= prosp_total*0.67 and prosp_total > 0 \
               else (r"$\sim$" if prosp_strong > 0 else r"\texttimes")
    lines.append(
        rf"— & Prospective prediction ($R^2>0.8$) & "
        rf"Exp E: {prosp_strong}/{prosp_total} (model,level) pairs $R^2>0.8$ & {prosp_sym} \\"
    )

    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)

tex_pred = make_predictions_table()
(OUT / "table_predictions_summary.tex").write_text(tex_pred, encoding="utf-8")
print("  [tex] table_predictions_summary.tex")

# ── LaTeX: PROSPECTIVE DETAIL TABLE ──────────────────────────────────────────

lines2 = [
    "% AUTO-GENERATED by part8_merge_figures.py",
    r"\begin{table}[t]",
    r"\centering\small",
    r"\caption{Experiment E prospective prediction results. "
    r"g-model fitted on $T\in\{150,300,600,1200\}$, "
    r"evaluated at $T\in\{2400,4800,9600\}$ without refitting. "
    r"$R^2_{\mathrm{test}}>0.8$ indicates the model captures a "
    r"real phenomenon rather than overfitting.}",
    r"\label{tab:prospective-detail}",
    r"\begin{tabular}{llccc}",
    r"\toprule",
    r"Model & Level & $R^2_{\mathrm{train}}$ & $R^2_{\mathrm{test}}$ & MAE (pp) \\",
    r"\midrule",
]
for mk in ["phi35","qwen3b"]:
    for li, lv in enumerate(["Level_1","Level_3","Level_5"]):
        res    = R7.get(mk,{}).get(lv,{})
        fit_r  = res.get("fit",{}) or {}
        eval_r = res.get("eval",{}) or {}
        mlbl   = MODEL_LBLS[mk] if li == 0 else ""
        lines2.append(
            rf"{mlbl} & {lv.replace('_',' ')} & "
            rf"${tex(fit_r.get('R2_train'))}$ & "
            rf"${tex(eval_r.get('R2_test'))}$ & "
            rf"${tex(eval_r.get('MAE'),'.1f')}$ \\"
        )
    lines2.append(r"\midrule")
lines2[-1] = r"\bottomrule"
lines2 += [r"\end{tabular}", r"\end{table}"]
(OUT / "table_prospective_summary.tex").write_text("\n".join(lines2), encoding="utf-8")
print("  [tex] table_prospective_summary.tex")

# ── FINAL PRINT SUMMARY ───────────────────────────────────────────────────────

print("\n" + "="*68)
print("  FULL EMPIRICAL PACKAGE SUMMARY")
print("="*68)

print("\n  P1 (T* monotone with difficulty):")
for mk in ["phi35","qwen3b","gemma2"]:
    p1 = R3.get("P1_per_model",{}).get(mk,{})
    if p1:
        t1 = p1.get("Level_1",0); t3=p1.get("Level_3",0); t5=p1.get("Level_5",0)
        ok = p1.get("monotonic","?")
        print(f"    {MODEL_LBLS[mk]:22s}  L1={t1:.0f}  L3={t3:.0f}  L5={t5:.0f}  "
              f"{'SUPPORTED' if ok else 'NOT SUPPORTED'}")

print("\n  P3 (T* decreases with quantisation):")
for lv in ["Level_1","Level_3","Level_5"]:
    p3t = R6.get("P3_tests",{}).get(lv,{})
    if p3t:
        ts = p3t.get("T_stars",{})
        row = "  ".join(f"{p}={ts.get(p,float('nan')):.0f}" for p in ["fp16","bf16","int8","int4"] if p in ts)
        print(f"    {lv}: {row}  "
              f"{'SUPPORTED' if p3t.get('monotone_decreasing') else 'NOT SUPPORTED'}")

print("\n  P5 (Hurst H≈0.5):")
for mk in ["phi35","qwen3b"]:
    for lv in ["Level_1","Level_3","Level_5"]:
        hr = R5.get(mk,{}).get(lv,{}).get("hurst",{})
        if "H" in hr:
            ci = hr.get("H_CI_boot",["?","?"])
            print(f"    {MODEL_LBLS[mk]:20s} {lv}: H={hr['H']:.3f} "
                  f"[{ci[0]:.3f},{ci[1]:.3f}]  R²={hr.get('R2',0):.3f}  "
                  f"{'✓' if hr.get('supports_P5') else '✗'}")

print("\n  P6 (T* decreases with N):")
for lv in ["Level_1","Level_3","Level_5"]:
    p6t = R4.get("P6_tests",{}).get(lv,{})
    if p6t:
        ts = p6t.get("T_stars",{})
        row = "  ".join(f"{MODEL_LBLS.get(mk,mk)}={ts[mk]:.0f}" for mk in ts)
        print(f"    {lv}: {row}  "
              f"{'SUPPORTED' if p6t.get('monotone_decreasing_with_N') else 'NOT SUPPORTED'}")

print("\n  Prospective prediction (R²_test):")
for mk in ["phi35","qwen3b"]:
    for lv in ["Level_1","Level_3","Level_5"]:
        r2t = R7.get(mk,{}).get(lv,{}).get("eval",{}).get("R2_test", float("nan"))
        if not math.isnan(r2t):
            mae = R7.get(mk,{}).get(lv,{}).get("eval",{}).get("MAE", float("nan"))
            strength = "STRONG" if r2t > 0.8 else ("MODERATE" if r2t > 0.5 else "WEAK")
            print(f"    {MODEL_LBLS[mk]:20s} {lv}: R²={r2t:.3f}  MAE={mae:.1f}pp  → {strength}")

print(f"\n  Outputs: {OUT}")
for fp in sorted(OUT.iterdir()):
    print(f"    {fp.name:50s}  {fp.stat().st_size/1024:7.1f} KB")
print("\n  DONE — all paper-ready outputs generated.")

---
## Done — Download your outputs

All figures, tables, and JSON result files are saved under `/kaggle/working/results/`.  
Use the Kaggle output panel (right sidebar) to download any file.

| Path | Contents |
|---|---|
| `results/expA/` | Exp A figures + `multimodel_results_expA.json` |
| `results/expB/` | Exp B figures + `p6_results_expB.json` |
| `results/expC/` | Exp C figures + `hurst_results_expC.json` |
| `results/expD/` | Exp D figures + `precision_results_expD.json` |
| `results/expE/` | Exp E figures + `prospective_results_expE.json` |
| `results/merged/` | Summary figures + LaTeX tables for paper |
